# NB-11_salidas_cnf_anual

Process Flow SAS: **Salidas_CNF_anual** — `ProcessFlowContainer-hhi2y41FQnpmRZqv`

In [ ]:
# ========= Parámetros =========
# Variables macro del SAS original. El .egp NO las define (venían del
# entorno SAS): su valor sale de la entrevista B4 o de
# project_config.yaml → run.macro_params, o se inyecta acá
# (celda 'parameters' de papermill).

Year = None  # &Year — nadie declaró su valor

faltantes = [n for n, v in {"Year": Year}.items() if v is None]
if faltantes:
    raise ValueError(f"Parámetros sin valor: {faltantes}")

In [ ]:
# ========= Celda 1: Configuración =========
import pandas as pd
import numpy as np
import os
import sqlalchemy
import datetime
from pathlib import Path
from sqlalchemy import text

# Conexión a BD — la define el orquestador al ejecutar
engine = sqlalchemy.create_engine(os.environ["SASMIG_DB_URL"], pool_pre_ping=True)
# Sesión de BD del notebook — espejo de la sesión WORK de SAS: las
# tablas temporales #tmp viven en ESTA conexión y mueren al cerrar el
# kernel. AUTOCOMMIT: cada statement commitea, como los pasos de SAS.
work_conn = engine.connect().execution_options(isolation_level="AUTOCOMMIT")

# Logging liviano de resultados — aprobado en la entrevista (Fase 4)
_LOG_PATH = Path("log") / "NB-11_salidas_cnf_anual.log"
_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
def _log(label, value=None):
    """Una línea por celda: imprime y persiste. Jamás rompe la corrida."""
    try:
        if hasattr(value, "shape"):
            detail = f"{value.shape[0]} filas x {value.shape[1]} cols"
        elif isinstance(value, int):
            detail = f"{value} filas"
        elif value is None:
            detail = "ok"
        else:
            detail = str(value)
        line = f"[{datetime.datetime.now():%Y-%m-%d %H:%M:%S}] {label}: {detail}"
        print(line)
        with open(_LOG_PATH, "a", encoding="utf-8") as fh:
            fh.write(line + "\n")
    except Exception:
        pass  # el log nunca puede tumbar el notebook
with open(_LOG_PATH, "a", encoding="utf-8") as _fh:
    _fh.write(f"\n=== corrida {datetime.datetime.now():%Y-%m-%d %H:%M:%S} ===\n")


## 1- TCons Mixtas_M_CGR

Redistribuye la muestra de transacciones mixtas (M) del año según la estructura de ponderación y expande el resto fuera de muestra, generando las bases desagregadas (mensual) y agregadas (trimestral/anual) para COFOG y cuentas nacionales

*confianza: medium · verificador: approve · SAS: PROC SQL multi-step: filtro inicial, múltiples LEFT JOIN con 4-6 condiciones de unión, agregaciones GROUP BY (incluyendo CALCULATED), UNION ALL (DATA step SET) y DELETE de ajuste*

In [ ]:
# ========= 1- TCons Mixtas_M_CGR =========
# /*FILTRA DATOS DE EJECUCION CGR DEL AÑO*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cgr_y"))
sql_ejec_cgr_y = f"""
SELECT *
INTO #ejec_cgr_y
FROM GOBGENER.dbo.EJECUCION
WHERE AÑO = {int(Year)}
"""
res = work_conn.execute(text(sql_ejec_cgr_y))
# DATA step DROP=: columnas de apoyo que no se usan en el resto del flujo
work_conn.execute(text(
    "ALTER TABLE #ejec_cgr_y DROP COLUMN "
    "PPTO_INICIAL, PPTO_ACTUAL, PERCIB_PAG, POR_PERCIB_PAG, N_ITEM, N_ASIGNACION, C_SUBASIGNACION, N_SUBASIGNACION"
))
_log("CREATE #ejec_cgr_y", res.rowcount)


In [ ]:
# /* CREA TABLA DE REDISTRIBUCION DE MUESTRA DE TRANSACCIONES MIXTAS (M) SEGUN ESTRUCTURA, PARA EJECUCION AÑO t */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mixtas_m"))
sql_ejec_muestra_mixtas_m = """
SELECT
    T2.SUBSECTOR, T2.MONEDA, T2.MES, T2.AÑO, T1.ENTIDAD AS C_ENTIDAD,
    T2.C_PARTIDA, T2.N_PARTIDA, T2.C_CAPITULO, T2.N_CAPITULO, T2.C_PROGRAMA, T2.N_PROGRAMA,
    T2.C_TIPO_CUENTA, T2.C_CUENTA, T2.N_CUENTA, T2.C_ITEM, T2.C_ASIGNACION, T2.C_ANALITICO, T2.N_ANALITICO,
    T1.C_SCN, T1.C_CAGENTE, T1.C_CUP,
    SUM(T2.DEVENG / 1000000.0 * T1.POND) AS DEVENGO
INTO #ejec_muestra_mixtas_m
FROM GOBGENER.dbo.EST_MIXTAS_M T1
LEFT JOIN #ejec_cgr_y T2
    ON T2.C_ENTIDAD = T1.ENTIDAD AND T2.C_ASIGNACION = T1.C_ASIGNACION
   AND T2.C_ANALITICO = T1.C_ANALITICO AND T2.C_CUENTA = T1.C_CUENTA
   AND T2.C_ITEM = T1.C_ITEM AND T2.MES = T1.MES
WHERE T2.C_CUENTA IN ('24', '33') AND T2.C_ITEM IN ('03', '01', '06', '08', '09')
  AND T1.ENTIDAD NOT IN (5601, 10201) AND T2.DEVENG IS NOT NULL
GROUP BY
    T2.SUBSECTOR, T2.MONEDA, T2.MES, T2.AÑO, T1.ENTIDAD,
    T2.C_PARTIDA, T2.N_PARTIDA, T2.C_CAPITULO, T2.N_CAPITULO, T2.C_PROGRAMA, T2.N_PROGRAMA,
    T2.C_TIPO_CUENTA, T2.C_CUENTA, T2.N_CUENTA, T2.C_ITEM, T2.C_ASIGNACION, T2.C_ANALITICO, T2.N_ANALITICO,
    T1.C_SCN, T1.C_CAGENTE, T1.C_CUP, T2.DEVENG
"""
res = work_conn.execute(text(sql_ejec_muestra_mixtas_m))
_log("CREATE #ejec_muestra_mixtas_m", res.rowcount)


In [ ]:
# /* CREA TABLA DE ANALITICOS DE MIXTAS (M) A EXPANDIR (FUERA DE MUESTRA), SEGUN EJECUCION AÑO t */
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_ejec_exp_mixtas_m"))
sql_cruce_ejec_exp_mixtas_m = """
SELECT T2.*, T1.Clasif_Tcons
INTO #cruce_ejec_exp_mixtas_m
FROM GOBGENER.dbo.ANALIT_TCONS T1
LEFT JOIN #ejec_cgr_y T2
    ON T2.C_ENTIDAD = T1.ENTIDAD AND T2.C_ASIGNACION = T1.C_ASIGNACION
   AND T2.C_ANALITICO = T1.C_ANALITICO AND T2.C_CUENTA = T1.C_CUENTA AND T2.C_ITEM = T1.C_ITEM
WHERE T1.M_MUESTRA = 0 AND T2.C_ENTIDAD NOT IN (5601, 10201) AND T1.Clasif_Tcons = 'M'
  AND T2.DEVENG IS NOT NULL
"""
res = work_conn.execute(text(sql_cruce_ejec_exp_mixtas_m))
_log("CREATE #cruce_ejec_exp_mixtas_m", res.rowcount)


In [ ]:
# /* CREA TABLA DE REDISTRIBUCION EXPANDIDA SEGUN ESTRUCTURA DE MUESTRA, PARA RESTO DE TRANSACCIONES MIXTAS (M), PARA EJECUCION AÑO t */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mixtas_m_prelim"))
sql_ejec_exp_mixtas_m_prelim = """
SELECT
    T2.SUBSECTOR, T2.MONEDA, T2.AÑO, T2.MES, T2.C_ENTIDAD,
    T2.C_PARTIDA, T2.N_PARTIDA, T2.C_CAPITULO, T2.N_CAPITULO, T2.C_PROGRAMA, T2.N_PROGRAMA,
    T2.C_TIPO_CUENTA, T2.C_CUENTA, T2.N_CUENTA, T2.C_ITEM, T2.C_ASIGNACION, T2.C_ANALITICO, T2.N_ANALITICO,
    T2.DEVENG, T2.Clasif_Tcons, T1.C_SCN, T1.C_CAGENTE, T1.C_CUP,
    SUM(T2.DEVENG) AS DATO_OR,
    SUM(T2.DEVENG / 1000000.0 * T1.POND * T3.DATO) AS DEVENGO
INTO #ejec_exp_mixtas_m_prelim
FROM GOBGENER.dbo.EST_EXP_MIXTAS_M T1
LEFT JOIN #cruce_ejec_exp_mixtas_m T2
    ON T2.C_CUENTA = T1.C_CUENTA AND T2.C_ITEM = T1.C_ITEM AND T2.C_ASIGNACION = T1.C_ASIGNACION
   AND T2.C_ANALITICO = T1.C_ANALITICO AND T2.MES = T1.MES AND T2.C_ENTIDAD = T1.ENTIDAD
LEFT JOIN #prom_mes T3
    ON T2.MONEDA = T3.MONEDA AND T2.MES = T3.MES AND T2.AÑO = T3.AÑO
WHERE T2.DEVENG IS NOT NULL
GROUP BY
    T2.SUBSECTOR, T2.MONEDA, T2.AÑO, T2.MES, T2.C_ENTIDAD,
    T2.C_PARTIDA, T2.N_PARTIDA, T2.C_CAPITULO, T2.N_CAPITULO, T2.C_PROGRAMA, T2.N_PROGRAMA,
    T2.C_TIPO_CUENTA, T2.C_CUENTA, T2.N_CUENTA, T2.C_ITEM, T2.C_ASIGNACION, T2.C_ANALITICO, T2.N_ANALITICO,
    T2.DEVENG, T2.Clasif_Tcons, T1.C_SCN, T1.C_CAGENTE, T1.C_CUP
"""
res = work_conn.execute(text(sql_ejec_exp_mixtas_m_prelim))
_log("CREATE #ejec_exp_mixtas_m_prelim", res.rowcount)


In [ ]:
# DATA EJEC_EXP_MIXTAS_M; SET EJEC_EXP_MIXTAS_M_prelim (DROP=DEVENG DATO_OR Clasif_Tcons); RUN;
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mixtas_m"))
res = work_conn.execute(text("SELECT * INTO #ejec_exp_mixtas_m FROM #ejec_exp_mixtas_m_prelim"))
work_conn.execute(text("ALTER TABLE #ejec_exp_mixtas_m DROP COLUMN DEVENG, DATO_OR, Clasif_Tcons"))
_log("CREATE #ejec_exp_mixtas_m", res.rowcount)


In [ ]:
# /* Crea tabla con la ejecución total de transacciones Mixtas (M), Muestra + Resto */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_mixtas_m_total"))
sql_ejec_mixtas_m_total = """
SELECT SUBSECTOR, MONEDA, AÑO, MES, C_ENTIDAD, C_PARTIDA, N_PARTIDA, C_CAPITULO, N_CAPITULO,
       C_PROGRAMA, N_PROGRAMA, C_TIPO_CUENTA, C_CUENTA, N_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO,
       N_ANALITICO, C_SCN, C_CAGENTE, C_CUP, DEVENGO
INTO #ejec_mixtas_m_total
FROM #ejec_muestra_mixtas_m
UNION ALL
SELECT SUBSECTOR, MONEDA, AÑO, MES, C_ENTIDAD, C_PARTIDA, N_PARTIDA, C_CAPITULO, N_CAPITULO,
       C_PROGRAMA, N_PROGRAMA, C_TIPO_CUENTA, C_CUENTA, N_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO,
       N_ANALITICO, C_SCN, C_CAGENTE, C_CUP, DEVENGO
FROM #ejec_exp_mixtas_m
"""
res = work_conn.execute(text(sql_ejec_mixtas_m_total))
_log("CREATE #ejec_mixtas_m_total", res.rowcount)


In [ ]:
# /* Ajuste por imputación de D63 en pandemia (2020) */
res = work_conn.execute(text(
    "DELETE FROM #ejec_mixtas_m_total "
    "WHERE C_ENTIDAD = 911 AND C_CUENTA = '24' AND C_ITEM = '03' AND C_ASIGNACION = '002' "
    "AND C_ANALITICO IN ('03002000') AND AÑO = 2020 AND MES > 4"
))
_log("DELETE #ejec_mixtas_m_total (ajuste D63 pandemia 2020)", res.rowcount)


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MIXTAS (M) para COFOG y ANALISIS DESAGREGADO CNF anual */
# /*Desagregado mes*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_e_mixt_aper"))
sql_ejec_e_mixt_aper = """
SELECT
    T1.SUBSECTOR, T1.MONEDA, T1.AÑO,
    CASE WHEN T1.MES IN (1,2,3) THEN 1
         WHEN T1.MES IN (4,5,6) THEN 2
         WHEN T1.MES IN (7,8,9) THEN 3
         ELSE 4 END AS [TRIM],
    T1.MES, T1.C_ENTIDAD, T1.C_PARTIDA, T1.C_CAPITULO, T1.C_PROGRAMA, T1.C_TIPO_CUENTA,
    T1.C_CUENTA, T1.C_ITEM, T1.C_ASIGNACION, T1.C_ANALITICO, T1.N_ANALITICO, T1.C_SCN,
    T1.C_CAGENTE, T1.C_CUP, T3.ACTIVIDAD, T3.NIVEL, T4.C_COFOG_1, T4.N_COFOG_1,
    SUM(T1.DEVENGO) AS Dato
INTO #ejec_e_mixt_aper
FROM #ejec_mixtas_m_total T1
LEFT JOIN GOBGENER.dbo.CR18_APERTURA T3 ON T1.C_ENTIDAD = T3.ENTIDAD
LEFT JOIN GOBGENER.dbo.COFOG T4
    ON T1.C_ENTIDAD = T4.ENTIDAD AND T1.C_PARTIDA = T4.C_PARTIDA
   AND T1.C_CAPITULO = T4.C_CAPITULO AND T1.C_PROGRAMA = T4.C_PROGRAMA
WHERE T1.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
GROUP BY
    T1.SUBSECTOR, T1.MONEDA, T1.AÑO,
    CASE WHEN T1.MES IN (1,2,3) THEN 1
         WHEN T1.MES IN (4,5,6) THEN 2
         WHEN T1.MES IN (7,8,9) THEN 3
         ELSE 4 END,
    T1.MES, T1.C_ENTIDAD, T1.C_PARTIDA, T1.C_CAPITULO, T1.C_PROGRAMA, T1.C_TIPO_CUENTA,
    T1.C_CUENTA, T1.C_ITEM, T1.C_ASIGNACION, T1.C_ANALITICO, T1.N_ANALITICO, T1.C_SCN,
    T1.C_CAGENTE, T1.C_CUP, T3.ACTIVIDAD, T3.NIVEL, T4.C_COFOG_1, T4.N_COFOG_1
"""
res = work_conn.execute(text(sql_ejec_e_mixt_aper))
_log("CREATE #ejec_e_mixt_aper", res.rowcount)


In [ ]:
# /*SALIDA DESAGREGADA ANUAL PARA COFOG*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_e_mixt_aper_an"))
sql_ejec_e_mixt_aper_an = """
SELECT
    SUBSECTOR, MONEDA, AÑO, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, C_TIPO_CUENTA,
    C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, C_SCN, C_CAGENTE, C_CUP,
    ACTIVIDAD, NIVEL, C_COFOG_1, N_COFOG_1,
    SUM(Dato) AS Dato
INTO #ejec_e_mixt_aper_an
FROM #ejec_e_mixt_aper
GROUP BY
    SUBSECTOR, MONEDA, AÑO, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, C_TIPO_CUENTA,
    C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, C_SCN, C_CAGENTE, C_CUP,
    ACTIVIDAD, NIVEL, C_COFOG_1, N_COFOG_1
"""
res = work_conn.execute(text(sql_ejec_e_mixt_aper_an))
_log("CREATE #ejec_e_mixt_aper_an", res.rowcount)


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MIXTAS (M) para CNF anual, AGREGADO*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_e_mixt"))
sql_ejec_e_mixt = """
SELECT
    SUBSECTOR, MONEDA, AÑO, [TRIM], MES, C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_SCN, C_CAGENTE, C_CUP,
    ACTIVIDAD, NIVEL, SUM(Dato) AS Dato
INTO #ejec_e_mixt
FROM #ejec_e_mixt_aper
GROUP BY
    SUBSECTOR, MONEDA, AÑO, [TRIM], MES, C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_SCN, C_CAGENTE, C_CUP,
    ACTIVIDAD, NIVEL
"""
res = work_conn.execute(text(sql_ejec_e_mixt))
_log("CREATE #ejec_e_mixt", res.rowcount)


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MIXTAS (M) para CNF anual, AGREGADA */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_mixtas_m_an"))
sql_ejec_mixtas_m_an = """
SELECT
    SUBSECTOR, MONEDA, AÑO, C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_SCN, C_CAGENTE, C_CUP,
    ACTIVIDAD, NIVEL, SUM(Dato) AS Dato
INTO #ejec_mixtas_m_an
FROM #ejec_e_mixt_aper
GROUP BY
    SUBSECTOR, MONEDA, AÑO, C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_SCN, C_CAGENTE, C_CUP,
    ACTIVIDAD, NIVEL
"""
res = work_conn.execute(text(sql_ejec_mixtas_m_an))
_log("CREATE #ejec_mixtas_m_an", res.rowcount)


In [ ]:
# PROC SQL; DROP TABLE EJEC_MUESTRA_MIXTAS_M, CRUCE_EJEC_EXP_MIXTAS_M, EJEC_EXP_MIXTAS_M_prelim, EJEC_EXP_MIXTAS_M, EJEC_MIXTAS_M_TOTAL; QUIT;
for t in ["#ejec_muestra_mixtas_m", "#cruce_ejec_exp_mixtas_m", "#ejec_exp_mixtas_m_prelim", "#ejec_exp_mixtas_m", "#ejec_mixtas_m_total"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


## 2- TCons MGCG_CGR

Redistribuye la ejecucion de transacciones MGCG entre la muestra de la CGR y el resto expandido por ponderacion, y la consolida por apertura COFOG y por agregados anuales para el analisis de cuentas nacionales

*confianza: medium · verificador: approve · SAS: PROC SQL: redistribucion de muestra y expansion de transacciones MGCG (LEFT JOINs + GROUP BY ponderado + UNION ALL) con salida agregada a nivel COFOG y CNF anual*

In [ ]:
# ========= 2- TCons MGCG_CGR =========
# /* CREA TABLA DE REDISTRIBUCION DE MUESTRA DE TRANSACCIONES MGCG SEGUN ESTRUCTURA, PARA EJECUCION AÑO t */
work_conn.execute(text('DROP TABLE IF EXISTS #ejec_muestra_mgcg_raw'))
sql_ejec_muestra_mgcg_raw = '''
SELECT
    T2.SUBSECTOR,
    T2.MONEDA,
    T2.MES,
    T2.AÑO,
    t1.ENTIDAD AS C_ENTIDAD,
    T2.C_PARTIDA,
    T2.N_PARTIDA,
    T2.C_CAPITULO,
    T2.N_CAPITULO,
    T2.C_PROGRAMA,
    T2.N_PROGRAMA,
    T2.C_TIPO_CUENTA,
    T2.C_CUENTA,
    T2.N_CUENTA,
    T2.C_ITEM,
    T2.C_ASIGNACION,
    T2.C_ANALITICO,
    T2.N_ANALITICO,
    T2.DEVENG,
    t1.C_SCN,
    t1.C_CAGENTE,
    SUM(T2.DEVENG / 1000000.0 * t1.POND) AS DEVENGO
INTO #ejec_muestra_mgcg_raw
FROM GOBGENER.dbo.EST_MGCG t1
LEFT JOIN #ejec_cgr_y t2
    ON t2.C_ENTIDAD = t1.ENTIDAD
   AND t2.C_ASIGNACION = t1.C_ASIGNACION
   AND t2.C_ANALITICO = t1.C_ANALITICO
   AND t2.C_CUENTA = t1.C_CUENTA
   AND t2.C_ITEM = t1.C_ITEM
   AND t2.MES = t1.MES
WHERE t2.C_CUENTA IN ('24')
  AND t2.C_ITEM IN ('03')
  AND t1.ENTIDAD NOT IN (5601, 10201)
  AND t2.DEVENG IS NOT NULL
GROUP BY
    T2.SUBSECTOR, T2.MONEDA, T2.MES, T2.AÑO, t1.ENTIDAD,
    T2.C_PARTIDA, T2.N_PARTIDA, T2.C_CAPITULO, T2.N_CAPITULO,
    T2.C_PROGRAMA, T2.N_PROGRAMA, T2.C_TIPO_CUENTA, T2.C_CUENTA,
    T2.N_CUENTA, T2.C_ITEM, T2.C_ASIGNACION, T2.C_ANALITICO,
    T2.N_ANALITICO, T2.DEVENG, t1.C_SCN, t1.C_CAGENTE
'''
work_conn.execute(text(sql_ejec_muestra_mgcg_raw))

# DATA EJEC_MUESTRA_MGCG; SET EJEC_MUESTRA_MGCG (DROP=DEVENG); -> columnas explicitas sin DEVENG, en el orden canonico compartido con EJEC_EXP_MGCG (clave para el UNION ALL posterior)
work_conn.execute(text('DROP TABLE IF EXISTS #ejec_muestra_mgcg'))
sql_ejec_muestra_mgcg = '''
SELECT
    SUBSECTOR, MONEDA, AÑO, MES, C_ENTIDAD, C_PARTIDA, N_PARTIDA,
    C_CAPITULO, N_CAPITULO, C_PROGRAMA, N_PROGRAMA, C_TIPO_CUENTA,
    C_CUENTA, N_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO,
    C_SCN, C_CAGENTE, DEVENGO
INTO #ejec_muestra_mgcg
FROM #ejec_muestra_mgcg_raw
'''
res = work_conn.execute(text(sql_ejec_muestra_mgcg))
work_conn.execute(text('DROP TABLE IF EXISTS #ejec_muestra_mgcg_raw'))
_log('CREATE #ejec_muestra_mgcg', res.rowcount)


In [ ]:
# /* CREA TABLA DE ANALITICOS DE MGCG A EXPANDIR (FUERA DE MUESTRA), SEGUN EJECUCION AÑO t */
work_conn.execute(text('DROP TABLE IF EXISTS #cruce_ejec_exp_mgcg'))
sql_cruce_ejec_exp_mgcg = '''
SELECT
    t2.SUBSECTOR, t2.MONEDA, t2.MES, t2.AÑO, t2.C_ENTIDAD, t2.C_PARTIDA, t2.N_PARTIDA,
    t2.C_CAPITULO, t2.N_CAPITULO, t2.C_PROGRAMA, t2.N_PROGRAMA, t2.C_TIPO_CUENTA,
    t2.C_CUENTA, t2.N_CUENTA, t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO, t2.N_ANALITICO,
    t2.DEVENG,
    t1.Clasif_Tcons
INTO #cruce_ejec_exp_mgcg
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN #ejec_cgr_y t2
    ON t2.C_ENTIDAD = t1.ENTIDAD
   AND t2.C_ASIGNACION = t1.C_ASIGNACION
   AND t2.C_ANALITICO = t1.C_ANALITICO
   AND t2.C_CUENTA = t1.C_CUENTA
   AND t2.C_ITEM = t1.C_ITEM
WHERE t1.M_MUESTRA = 0
  AND t1.Clasif_Tcons = 'MGCG'
  AND t2.C_ENTIDAD NOT IN (5601, 10201)
  AND t2.DEVENG IS NOT NULL
'''
res = work_conn.execute(text(sql_cruce_ejec_exp_mgcg))
_log('CREATE #cruce_ejec_exp_mgcg', res.rowcount)


In [ ]:
# /* CREA TABLA DE REDISTRIBUCION EXPANDIDA SEGUN ESTRUCTURA DE MUESTRA, PARA RESTO DE TRANSACCIONES MGCG, PARA EJECUCION AÑO t */
work_conn.execute(text('DROP TABLE IF EXISTS #ejec_exp_mgcg_prelim'))
sql_ejec_exp_mgcg_prelim = '''
SELECT
    t2.SUBSECTOR, t2.MONEDA, t2.AÑO, t2.MES, t2.C_ENTIDAD, t2.C_PARTIDA, t2.N_PARTIDA,
    t2.C_CAPITULO, t2.N_CAPITULO, t2.C_PROGRAMA, t2.N_PROGRAMA, t2.C_TIPO_CUENTA,
    t2.C_CUENTA, t2.N_CUENTA, t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO, t2.N_ANALITICO,
    t2.DEVENG, t2.Clasif_Tcons,
    t1.C_SCN, t1.C_CAGENTE,
    SUM(t2.DEVENG) AS DATO_OR,
    SUM(t2.DEVENG / 1000000.0 * t1.POND * t3.DATO) AS DEVENGO
INTO #ejec_exp_mgcg_prelim
FROM GOBGENER.dbo.EST_EXP_MGCG t1
LEFT JOIN #cruce_ejec_exp_mgcg t2
    ON t2.C_CUENTA = t1.C_CUENTA
   AND t2.C_ITEM = t1.C_ITEM
   AND t2.MES = t1.MES
LEFT JOIN #prom_mes t3
    ON t2.MONEDA = t3.MONEDA
   AND t2.MES = t3.MES
   AND t2.AÑO = t3.AÑO
WHERE t2.DEVENG IS NOT NULL
GROUP BY
    t2.SUBSECTOR, t2.MONEDA, t2.AÑO, t2.MES, t2.C_ENTIDAD,
    t2.C_PARTIDA, t2.N_PARTIDA, t2.C_CAPITULO, t2.N_CAPITULO,
    t2.C_PROGRAMA, t2.N_PROGRAMA, t2.C_TIPO_CUENTA, t2.C_CUENTA,
    t2.N_CUENTA, t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO,
    t2.N_ANALITICO, t2.DEVENG, t2.Clasif_Tcons, t1.C_SCN, t1.C_CAGENTE
'''
work_conn.execute(text(sql_ejec_exp_mgcg_prelim))

# DATA EJEC_EXP_MGCG; SET EJEC_EXP_MGCG_prelim (DROP=DEVENG DATO_OR Clasif_Tcons); -> mismo orden canonico que EJEC_MUESTRA_MGCG, clave para el UNION ALL siguiente
work_conn.execute(text('DROP TABLE IF EXISTS #ejec_exp_mgcg'))
sql_ejec_exp_mgcg = '''
SELECT
    SUBSECTOR, MONEDA, AÑO, MES, C_ENTIDAD, C_PARTIDA, N_PARTIDA,
    C_CAPITULO, N_CAPITULO, C_PROGRAMA, N_PROGRAMA, C_TIPO_CUENTA,
    C_CUENTA, N_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO,
    C_SCN, C_CAGENTE, DEVENGO
INTO #ejec_exp_mgcg
FROM #ejec_exp_mgcg_prelim
'''
res = work_conn.execute(text(sql_ejec_exp_mgcg))
_log('CREATE #ejec_exp_mgcg', res.rowcount)


In [ ]:
# /* Crea tabla con la ejecucion total de transacciones MGCG, Muestra + Resto */
# UNION ALL por NOMBRE de columna explicito en ambos SELECT (no 'SELECT *' posicional): el SAS
# original (DATA EJEC_MGCG_TOTAL; SET EJEC_MUESTRA_MGCG EJEC_EXP_MGCG;) alinea por NOMBRE de
# variable, y EJEC_MUESTRA_MGCG trae MES antes que AÑO mientras EJEC_EXP_MGCG trae AÑO antes
# que MES; un UNION ALL posicional intercambiaria esos valores para la mitad de las filas.
work_conn.execute(text('DROP TABLE IF EXISTS #ejec_mgcg_total'))
_cols_mgcg_total = (
    'SUBSECTOR, MONEDA, AÑO, MES, C_ENTIDAD, C_PARTIDA, N_PARTIDA, '
    'C_CAPITULO, N_CAPITULO, C_PROGRAMA, N_PROGRAMA, C_TIPO_CUENTA, '
    'C_CUENTA, N_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, '
    'C_SCN, C_CAGENTE, DEVENGO'
)
sql_ejec_mgcg_total = f'''
SELECT {_cols_mgcg_total}
INTO #ejec_mgcg_total
FROM #ejec_muestra_mgcg
UNION ALL
SELECT {_cols_mgcg_total}
FROM #ejec_exp_mgcg
'''
res = work_conn.execute(text(sql_ejec_mgcg_total))
_log('CREATE #ejec_mgcg_total', res.rowcount)


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MGCG para COFOG y ANALISIS DESAGREGADO CNF anual */
# /*Desagregado mes*/
work_conn.execute(text('DROP TABLE IF EXISTS #ejec_e_mgcg_aper'))
sql_ejec_e_mgcg_aper = '''
SELECT
    t1.SUBSECTOR,
    t1.MONEDA,
    t1.AÑO,
    CASE WHEN t1.MES IN (1, 2, 3) THEN 1
         WHEN t1.MES IN (4, 5, 6) THEN 2
         WHEN t1.MES IN (7, 8, 9) THEN 3
         ELSE 4 END AS TRIM,
    t1.MES,
    t1.C_ENTIDAD,
    t1.C_PARTIDA,
    t1.C_CAPITULO,
    t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA,
    t1.C_CUENTA,
    t1.C_ITEM,
    t1.C_ASIGNACION,
    t1.C_ANALITICO,
    t1.N_ANALITICO,
    t1.C_SCN,
    t1.C_CAGENTE,
    t3.ACTIVIDAD,
    t3.NIVEL,
    t4.C_COFOG_1,
    t4.N_COFOG_1,
    SUM(t1.DEVENGO) AS DATO
INTO #ejec_e_mgcg_aper
FROM #ejec_mgcg_total t1
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t3 ON t1.C_ENTIDAD = t3.ENTIDAD
LEFT JOIN GOBGENER.dbo.COFOG t4
    ON t1.C_ENTIDAD = t4.ENTIDAD
   AND t1.C_PARTIDA = t4.C_PARTIDA
   AND t1.C_CAPITULO = t4.C_CAPITULO
   AND t1.C_PROGRAMA = t4.C_PROGRAMA
WHERE t1.C_SCN NOT IN ('F22', 'F29', 'F3', 'F32', 'F41', 'F42', 'F5', 'F612', 'F7', 'F71', 'F79', 'K10', 'K11', 'K9')
GROUP BY
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    CASE WHEN t1.MES IN (1, 2, 3) THEN 1
         WHEN t1.MES IN (4, 5, 6) THEN 2
         WHEN t1.MES IN (7, 8, 9) THEN 3
         ELSE 4 END,
    t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
    t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE,
    t3.ACTIVIDAD, t3.NIVEL, t4.C_COFOG_1, t4.N_COFOG_1
'''
res = work_conn.execute(text(sql_ejec_e_mgcg_aper))
_log('CREATE #ejec_e_mgcg_aper', res.rowcount)


In [ ]:
# /*Agregado año, SALIDA COFOG ANALISIS DESAGREGADO CNF anual */
work_conn.execute(text('DROP TABLE IF EXISTS #ejec_e_mgcg_aper_an'))
sql_ejec_e_mgcg_aper_an = '''
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t1.C_SCN, t1.C_CAGENTE, t1.ACTIVIDAD, t1.NIVEL, t1.C_COFOG_1, t1.N_COFOG_1,
    SUM(t1.DATO) AS DATO
INTO #ejec_e_mgcg_aper_an
FROM #ejec_e_mgcg_aper t1
GROUP BY
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t1.C_SCN, t1.C_CAGENTE, t1.ACTIVIDAD, t1.NIVEL, t1.C_COFOG_1, t1.N_COFOG_1
'''
res = work_conn.execute(text(sql_ejec_e_mgcg_aper_an))
_log('CREATE #ejec_e_mgcg_aper_an', res.rowcount)


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MGCG para CNF anual */
work_conn.execute(text('DROP TABLE IF EXISTS #ejec_e_mgcg'))
sql_ejec_e_mgcg = '''
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
    t1.C_SCN, t1.C_CAGENTE, SUM(t1.DATO) AS DATO, t1.ACTIVIDAD, t1.NIVEL
INTO #ejec_e_mgcg
FROM #ejec_e_mgcg_aper t1
GROUP BY
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
    t1.C_SCN, t1.C_CAGENTE, t1.ACTIVIDAD, t1.NIVEL
'''
res = work_conn.execute(text(sql_ejec_e_mgcg))
_log('CREATE #ejec_e_mgcg', res.rowcount)


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MGCG para CNF anual */
work_conn.execute(text('DROP TABLE IF EXISTS #ejec_mgcg_an'))
sql_ejec_mgcg_an = '''
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE,
    t1.ACTIVIDAD, t1.NIVEL, SUM(t1.DATO) AS DATO
INTO #ejec_mgcg_an
FROM #ejec_e_mgcg_aper t1
GROUP BY
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE,
    t1.ACTIVIDAD, t1.NIVEL
'''
res = work_conn.execute(text(sql_ejec_mgcg_an))
_log('CREATE #ejec_mgcg_an', res.rowcount)


In [ ]:
# PROC SQL; DROP TABLE EJEC_MUESTRA_MGCG, CRUCE_EJEC_EXP_MGCG, EJEC_EXP_MGCG_prelim, EJEC_EXP_MGCG, EJEC_MGCG_TOTAL;
for _t in ['#ejec_muestra_mgcg', '#cruce_ejec_exp_mgcg', '#ejec_exp_mgcg_prelim', '#ejec_exp_mgcg', '#ejec_mgcg_total']:
    work_conn.execute(text(f'DROP TABLE IF EXISTS {_t}'))


## 3- TCons MM_CGR

Redistribuye y expande la ejecución de transacciones MM entre muestra y resto según estructura y ponderadores, agregando por entidad/cuenta y por apertura funcional (COFOG) a nivel mensual y anual

*confianza: low · verificador: approve · SAS: PROC SQL: cascada de CREATE TABLE con LEFT JOIN + GROUP BY (redistribución de muestra y expansión) y UNION de DATA steps*

In [ ]:
# ========= 3- TCons MM_CGR =========
# /* CREA TABLA DE REDISTRIBUCION DE MUESTRA DE TRANSACCIONES MM SEGUN ESTRUCTURA, PARA EJECUCION AÑO t */
# PROC SQL CREATE TABLE + DATA step DROP=DEVENG fusionados en una sola escritura server-side
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mm"))
sql_ejec_muestra_mm = """
SELECT
    t2.SUBSECTOR,
    t2.MONEDA,
    t2.AÑO,
    t2.MES,
    t2.C_ENTIDAD,
    t2.C_PARTIDA,
    t2.N_PARTIDA,
    t2.C_CAPITULO,
    t2.N_CAPITULO,
    t2.C_PROGRAMA,
    t2.N_PROGRAMA,
    t2.C_TIPO_CUENTA,
    t2.C_CUENTA,
    t2.N_CUENTA,
    t2.C_ITEM,
    t2.C_ASIGNACION,
    t2.C_ANALITICO,
    t2.N_ANALITICO,
    t1.C_SCN,
    t1.C_CAGENTE,
    SUM(t2.DEVENG / 1000000.0 * t1.POND) AS DEVENGO
INTO #ejec_muestra_mm
FROM GOBGENER.dbo.EST_MM t1
LEFT JOIN #ejec_cgr_y t2
    ON t2.C_ENTIDAD = t1.ENTIDAD
   AND t2.C_ASIGNACION = t1.C_ASIGNACION
   AND t2.C_ANALITICO = t1.C_ANALITICO
   AND t2.C_CUENTA = t1.C_CUENTA
   AND t2.C_ITEM = t1.C_ITEM
   AND t2.MES = t1.MES
WHERE t2.C_CUENTA IN ('24', '33')
  AND t2.C_ITEM IN ('01', '03', '08', '09')
  AND t2.DEVENG IS NOT NULL
  AND t1.ENTIDAD NOT IN (5601, 10201)
GROUP BY
    t2.SUBSECTOR, t2.MONEDA, t2.AÑO, t2.MES, t2.C_ENTIDAD,
    t2.C_PARTIDA, t2.N_PARTIDA, t2.C_CAPITULO, t2.N_CAPITULO,
    t2.C_PROGRAMA, t2.N_PROGRAMA, t2.C_TIPO_CUENTA, t2.C_CUENTA,
    t2.N_CUENTA, t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO, t2.N_ANALITICO,
    t1.C_SCN, t1.C_CAGENTE, t2.DEVENG
"""
res = work_conn.execute(text(sql_ejec_muestra_mm))
_log("#ejec_muestra_mm", res.rowcount)


In [ ]:
# /* CREA TABLA DE ANALITICOS DE MM A EXPANDIR (FUERA DE MUESTRA), SEGUN EJECUCION AÑO t */
# Patrón SAS de 'última columna gana': ANALIT_TCONS (t1, tabla conductora del LEFT JOIN) pisa
# las claves de EJEC_CGR_Y (t2) para evitar nulos cuando no hay match
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_ejec_exp_mm"))
sql_cruce_ejec_exp_mm = """
SELECT
    t2.SUBSECTOR,
    t2.MONEDA,
    t2.AÑO,
    t2.MES,
    t2.C_ENTIDAD,
    t2.C_PARTIDA,
    t2.N_PARTIDA,
    t2.C_CAPITULO,
    t2.N_CAPITULO,
    t2.C_PROGRAMA,
    t2.N_PROGRAMA,
    t2.C_TIPO_CUENTA,
    t2.N_CUENTA,
    t2.N_ANALITICO,
    t2.DEVENG,
    t1.C_CUENTA,
    t1.C_ITEM,
    t1.ENTIDAD,
    t1.C_ASIGNACION,
    t1.C_ANALITICO,
    t1.Clasif_Tcons
INTO #cruce_ejec_exp_mm
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN #ejec_cgr_y t2
    ON t2.C_ENTIDAD = t1.ENTIDAD
   AND t2.C_ASIGNACION = t1.C_ASIGNACION
   AND t2.C_ANALITICO = t1.C_ANALITICO
   AND t2.C_CUENTA = t1.C_CUENTA
   AND t2.C_ITEM = t1.C_ITEM
WHERE t1.M_MUESTRA = 0
  AND t1.Clasif_Tcons = 'MM'
  AND t1.ENTIDAD NOT IN (5601, 10201)
  AND t2.DEVENG IS NOT NULL
"""
res = work_conn.execute(text(sql_cruce_ejec_exp_mm))
_log("#cruce_ejec_exp_mm", res.rowcount)


In [ ]:
# /* CREA TABLA DE REDISTRIBUCION EXPANDIDA SEGUN ESTRUCTURA DE MUESTRA, PARA RESTO DE TRANSACCIONES MM, PARA EJECUCION AÑO t */
# PROC SQL CREATE TABLE + DATA step DROP=DEVENG DATO_OR Clasif_Tcons fusionados;
# la columna ENTIDAD (vestigial, de t1.ENTIDAD en #cruce_ejec_exp_mm) no se usa en ningún
# JOIN ni consulta posterior y se omite (ver warnings)
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mm_prelim"))
sql_ejec_exp_mm_prelim = """
SELECT
    t2.SUBSECTOR,
    t2.MONEDA,
    t2.AÑO,
    t2.MES,
    t2.C_ENTIDAD,
    t2.C_PARTIDA,
    t2.N_PARTIDA,
    t2.C_CAPITULO,
    t2.N_CAPITULO,
    t2.C_PROGRAMA,
    t2.N_PROGRAMA,
    t2.C_TIPO_CUENTA,
    t2.C_CUENTA,
    t2.N_CUENTA,
    t2.C_ITEM,
    t2.C_ASIGNACION,
    t2.C_ANALITICO,
    t2.N_ANALITICO,
    t1.C_SCN,
    t1.C_CAGENTE,
    SUM(t2.DEVENG / 1000000.0 * t1.POND * t3.DATO) AS DEVENGO
INTO #ejec_exp_mm_prelim
FROM GOBGENER.dbo.EST_EXP_MM t1
LEFT JOIN #cruce_ejec_exp_mm t2
    ON t2.C_CUENTA = t1.C_CUENTA
   AND t2.C_ITEM = t1.C_ITEM
   AND t2.MES = t1.MES
   AND t2.C_ENTIDAD = t1.ENTIDAD
   AND t2.C_ASIGNACION = t1.C_ASIGNACION
   AND t2.C_ANALITICO = t1.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t2.MONEDA = t3.MONEDA
   AND t2.MES = t3.MES
   AND t2.AÑO = t3.AÑO
WHERE t2.DEVENG IS NOT NULL
GROUP BY
    t2.SUBSECTOR, t2.MONEDA, t2.AÑO, t2.MES, t2.C_ENTIDAD,
    t2.C_PARTIDA, t2.N_PARTIDA, t2.C_CAPITULO, t2.N_CAPITULO,
    t2.C_PROGRAMA, t2.N_PROGRAMA, t2.C_TIPO_CUENTA, t2.C_CUENTA,
    t2.N_CUENTA, t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO, t2.N_ANALITICO,
    t2.DEVENG, t2.Clasif_Tcons, t1.C_SCN, t1.C_CAGENTE
"""
res = work_conn.execute(text(sql_ejec_exp_mm_prelim))
_log("#ejec_exp_mm_prelim", res.rowcount)


In [ ]:
# /* Crea tabla con la ejecución total de transacciones MGG, Muestra + Resto */
# (comentario original del SAS dice MGG; este nodo procesa MM - se conserva literal, ver warnings)
# Orden EJEC_EXP_MM_prelim + EJEC_MUESTRA_MM igual que el SET original
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_mm_total"))
sql_ejec_mm_total = """
SELECT SUBSECTOR, MONEDA, AÑO, MES, C_ENTIDAD, C_PARTIDA, N_PARTIDA, C_CAPITULO, N_CAPITULO,
       C_PROGRAMA, N_PROGRAMA, C_TIPO_CUENTA, C_CUENTA, N_CUENTA, C_ITEM, C_ASIGNACION,
       C_ANALITICO, N_ANALITICO, C_SCN, C_CAGENTE, DEVENGO
INTO #ejec_mm_total
FROM (
    SELECT SUBSECTOR, MONEDA, AÑO, MES, C_ENTIDAD, C_PARTIDA, N_PARTIDA, C_CAPITULO, N_CAPITULO,
           C_PROGRAMA, N_PROGRAMA, C_TIPO_CUENTA, C_CUENTA, N_CUENTA, C_ITEM, C_ASIGNACION,
           C_ANALITICO, N_ANALITICO, C_SCN, C_CAGENTE, DEVENGO
    FROM #ejec_exp_mm_prelim
    UNION ALL
    SELECT SUBSECTOR, MONEDA, AÑO, MES, C_ENTIDAD, C_PARTIDA, N_PARTIDA, C_CAPITULO, N_CAPITULO,
           C_PROGRAMA, N_PROGRAMA, C_TIPO_CUENTA, C_CUENTA, N_CUENTA, C_ITEM, C_ASIGNACION,
           C_ANALITICO, N_ANALITICO, C_SCN, C_CAGENTE, DEVENGO
    FROM #ejec_muestra_mm
) u
"""
res = work_conn.execute(text(sql_ejec_mm_total))
_log("#ejec_mm_total", res.rowcount)


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MM para COFOG y ANALISIS DESAGREGADO CNF anual */
# /*Desagregado mes*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_e_mm_aper"))
sql_ejec_e_mm_aper = """
SELECT
    t1.SUBSECTOR,
    t1.MONEDA,
    t1.AÑO,
    CASE
        WHEN t1.MES IN (1, 2, 3) THEN 1
        WHEN t1.MES IN (4, 5, 6) THEN 2
        WHEN t1.MES IN (7, 8, 9) THEN 3
        ELSE 4
    END AS TRIM,
    t1.MES,
    t1.C_ENTIDAD,
    t1.C_PARTIDA,
    t1.C_CAPITULO,
    t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA,
    t1.C_CUENTA,
    t1.C_ITEM,
    t1.C_ASIGNACION,
    t1.C_ANALITICO,
    t1.N_ANALITICO,
    t1.C_SCN,
    t1.C_CAGENTE,
    t3.ACTIVIDAD,
    t3.NIVEL,
    t4.C_COFOG_1,
    t4.N_COFOG_1,
    SUM(t1.DEVENGO) AS DATO
INTO #ejec_e_mm_aper
FROM #ejec_mm_total t1
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t3 ON t1.C_ENTIDAD = t3.ENTIDAD
LEFT JOIN GOBGENER.dbo.COFOG t4
    ON t1.C_ENTIDAD = t4.ENTIDAD
   AND t1.C_PARTIDA = t4.C_PARTIDA
   AND t1.C_CAPITULO = t4.C_CAPITULO
   AND t1.C_PROGRAMA = t4.C_PROGRAMA
WHERE t1.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
GROUP BY
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    CASE
        WHEN t1.MES IN (1, 2, 3) THEN 1
        WHEN t1.MES IN (4, 5, 6) THEN 2
        WHEN t1.MES IN (7, 8, 9) THEN 3
        ELSE 4
    END,
    t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
    t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE, t3.ACTIVIDAD, t3.NIVEL, t4.C_COFOG_1, t4.N_COFOG_1
"""
res = work_conn.execute(text(sql_ejec_e_mm_aper))
_log("#ejec_e_mm_aper", res.rowcount)


In [ ]:
# /*Agregado año*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_e_mm_aper_an"))
sql_ejec_e_mm_aper_an = """
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
    t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
    t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE,
    t1.ACTIVIDAD, t1.NIVEL, t1.C_COFOG_1, t1.N_COFOG_1,
    SUM(t1.DATO) AS DATO
INTO #ejec_e_mm_aper_an
FROM #ejec_e_mm_aper t1
GROUP BY
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
    t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
    t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE,
    t1.ACTIVIDAD, t1.NIVEL, t1.C_COFOG_1, t1.N_COFOG_1
"""
res = work_conn.execute(text(sql_ejec_e_mm_aper_an))
_log("#ejec_e_mm_aper_an", res.rowcount)


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MM para CNF anual */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_e_mm"))
sql_ejec_e_mm = """
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA,
    t1.C_CUENTA, t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.ACTIVIDAD, t1.NIVEL,
    SUM(t1.DATO) AS DATO
INTO #ejec_e_mm
FROM #ejec_e_mm_aper t1
GROUP BY
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA,
    t1.C_CUENTA, t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.ACTIVIDAD, t1.NIVEL
"""
res = work_conn.execute(text(sql_ejec_e_mm))
_log("#ejec_e_mm", res.rowcount)


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MM para CNF anual */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_mm_an"))
sql_ejec_mm_an = """
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
    t1.C_SCN, t1.C_CAGENTE, t1.ACTIVIDAD, t1.NIVEL,
    SUM(t1.DATO) AS DATO
INTO #ejec_mm_an
FROM #ejec_e_mm_aper_an t1
GROUP BY
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
    t1.C_SCN, t1.C_CAGENTE, t1.ACTIVIDAD, t1.NIVEL
"""
res = work_conn.execute(text(sql_ejec_mm_an))
_log("#ejec_mm_an", res.rowcount)


In [ ]:
# PROC SQL DROP TABLE de limpieza final: libera las tablas de sesión intermedias, igual que el SAS original
for _t in ["#ejec_muestra_mm", "#cruce_ejec_exp_mm", "#ejec_exp_mm_prelim", "#ejec_exp_mm", "#ejec_mm_total"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


## 4- TCons MGG_CGR

Redistribuye las transacciones MGG (muestra y resto) según la estructura declarada, las cruza con apertura institucional y clasificador COFOG, y arma los resúmenes mensual y anual por cuenta y código SCN

*confianza: medium · verificador: approve · SAS: PROC SQL CREATE TABLE con múltiples LEFT JOIN y GROUP BY encadenados + DATA step SET/DROP, con limpieza final de tablas WORK*

In [ ]:
# ========= 4- TCons MGG_CGR =========
# /* CREA TABLA DE REDISTRIBUCION DE MUESTRA DE TRANSACCIONES MGG SEGUN ESTRUCTURA, PARA EJECUCION AÑO t */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mgg_prelim"))
sql_ejec_muestra_mgg_prelim = """
SELECT
    t2.SUBSECTOR, t2.MONEDA, t2.MES, t2.AÑO, t1.ENTIDAD AS C_ENTIDAD,
    t2.C_PARTIDA, t2.N_PARTIDA, t2.C_CAPITULO, t2.N_CAPITULO,
    t2.C_PROGRAMA, t2.N_PROGRAMA, t2.C_TIPO_CUENTA, t2.C_CUENTA,
    t2.N_CUENTA, t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO,
    t2.N_ANALITICO, t2.DEVENG, t1.C_SCN, t1.C_CAGENTE,
    SUM(t2.DEVENG / 1000000.0 * t1.POND) AS DEVENGO
INTO #ejec_muestra_mgg_prelim
FROM GOBGENER.dbo.EST_MGG t1
LEFT JOIN #ejec_cgr_y t2
    ON t2.C_ENTIDAD = t1.ENTIDAD
   AND t2.C_ASIGNACION = t1.C_ASIGNACION
   AND t2.C_ANALITICO = t1.C_ANALITICO
   AND t2.C_CUENTA = t1.C_CUENTA
   AND t2.C_ITEM = t1.C_ITEM
   AND t2.MES = t1.MES
WHERE t2.C_CUENTA IN ('24')
  AND t2.C_ITEM IN ('01','03','08','09')
  AND t1.ENTIDAD NOT IN (5601, 10201)
  AND t2.DEVENG IS NOT NULL
GROUP BY
    t2.SUBSECTOR, t2.MONEDA, t2.MES, t2.AÑO, t1.ENTIDAD,
    t2.C_PARTIDA, t2.N_PARTIDA, t2.C_CAPITULO, t2.N_CAPITULO,
    t2.C_PROGRAMA, t2.N_PROGRAMA, t2.C_TIPO_CUENTA, t2.C_CUENTA,
    t2.N_CUENTA, t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO,
    t2.N_ANALITICO, t2.DEVENG, t1.C_SCN, t1.C_CAGENTE
"""
res = work_conn.execute(text(sql_ejec_muestra_mgg_prelim))
_log("CREATE #ejec_muestra_mgg_prelim", res.rowcount)


In [ ]:
# DATA EJEC_MUESTRA_MGG; SET EJEC_MUESTRA_MGG_prelim (DROP=DEVENG); -> descarta el monto original, deja solo el redistribuido
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mgg"))
sql_ejec_muestra_mgg = """
SELECT
    SUBSECTOR, MONEDA, MES, AÑO, C_ENTIDAD, C_PARTIDA, N_PARTIDA,
    C_CAPITULO, N_CAPITULO, C_PROGRAMA, N_PROGRAMA, C_TIPO_CUENTA,
    C_CUENTA, N_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO,
    C_SCN, C_CAGENTE, DEVENGO
INTO #ejec_muestra_mgg
FROM #ejec_muestra_mgg_prelim
"""
res = work_conn.execute(text(sql_ejec_muestra_mgg))
_log("CREATE #ejec_muestra_mgg", res.rowcount)


In [ ]:
# /* CREA TABLA DE ANALITICOS DE MGG A EXPANDIR (FUERA DE MUESTRA), SEGUN EJECUCION AÑO t */
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_ejec_exp_mgg"))
# t1 (ANALIT_TCONS) aporta la clasificación a expandir; sus códigos
# (C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO) prevalecen sobre los
# homónimos de t2, igual que el remerge de SAS para columnas duplicadas
sql_cruce_ejec_exp_mgg = """
SELECT
    t2.SUBSECTOR, t2.MONEDA, t2.MES, t2.AÑO, t2.C_ENTIDAD,
    t2.C_PARTIDA, t2.N_PARTIDA, t2.C_CAPITULO, t2.N_CAPITULO,
    t2.C_PROGRAMA, t2.N_PROGRAMA, t2.C_TIPO_CUENTA,
    t2.N_CUENTA, t2.N_ANALITICO, t2.DEVENG,
    t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.Clasif_Tcons
INTO #cruce_ejec_exp_mgg
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN #ejec_cgr_y t2
    ON t2.C_ENTIDAD = t1.ENTIDAD
   AND t2.C_ASIGNACION = t1.C_ASIGNACION
   AND t2.C_ANALITICO = t1.C_ANALITICO
   AND t2.C_CUENTA = t1.C_CUENTA
   AND t2.C_ITEM = t1.C_ITEM
WHERE t1.M_MUESTRA = 0
  AND t1.Clasif_Tcons = 'MGG'
  AND t2.C_ENTIDAD NOT IN (5601, 10201)
  AND t2.DEVENG IS NOT NULL
"""
res = work_conn.execute(text(sql_cruce_ejec_exp_mgg))
_log("CREATE #cruce_ejec_exp_mgg", res.rowcount)


In [ ]:
# /* CREA TABLA DE REDISTRIBUCION EXPANDIDA SEGUN ESTRUCTURA DE MUESTRA, PARA RESTO DE TRANSACCIONES MGG, PARA EJECUCION AÑO t */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mgg_preliminar_raw"))
sql_ejec_exp_mgg_preliminar_raw = """
SELECT
    t2.SUBSECTOR, t2.MONEDA, t2.AÑO, t2.MES, t2.C_ENTIDAD,
    t2.C_PARTIDA, t2.N_PARTIDA, t2.C_CAPITULO, t2.N_CAPITULO,
    t2.C_PROGRAMA, t2.N_PROGRAMA, t2.C_TIPO_CUENTA, t2.C_CUENTA,
    t2.N_CUENTA, t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO,
    t2.N_ANALITICO, t2.DEVENG, t2.Clasif_Tcons,
    t1.C_SCN, t1.C_CAGENTE,
    SUM(t2.DEVENG) AS DATO_OR,
    SUM(t2.DEVENG / 1000000.0 * t1.POND * t3.DATO) AS DEVENGO
INTO #ejec_exp_mgg_preliminar_raw
FROM GOBGENER.dbo.EST_EXP_MGG t1
LEFT JOIN #cruce_ejec_exp_mgg t2
    ON t2.C_CUENTA = t1.C_CUENTA
   AND t2.C_ITEM = t1.C_ITEM
   AND t2.MES = t1.MES
LEFT JOIN #prom_mes t3
    ON t2.MONEDA = t3.MONEDA
   AND t2.MES = t3.MES
   AND t2.AÑO = t3.AÑO
WHERE t2.DEVENG IS NOT NULL
GROUP BY
    t2.SUBSECTOR, t2.MONEDA, t2.AÑO, t2.MES, t2.C_ENTIDAD,
    t2.C_PARTIDA, t2.N_PARTIDA, t2.C_CAPITULO, t2.N_CAPITULO,
    t2.C_PROGRAMA, t2.N_PROGRAMA, t2.C_TIPO_CUENTA, t2.C_CUENTA,
    t2.N_CUENTA, t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO,
    t2.N_ANALITICO, t2.DEVENG, t2.Clasif_Tcons, t1.C_SCN, t1.C_CAGENTE
"""
work_conn.execute(text(sql_ejec_exp_mgg_preliminar_raw))


In [ ]:
# DATA EJEC_EXP_MGG_preliminar; SET EJEC_EXP_MGG_preliminar (DROP=DEVENG DATO_OR Clasif_Tcons);
# se materializa con nombre propio (#ejec_exp_mgg_preliminar) porque SQL Server
# no permite un SELECT INTO que lea y escriba la misma tabla
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mgg_preliminar"))
sql_ejec_exp_mgg_preliminar = """
SELECT
    SUBSECTOR, MONEDA, AÑO, MES, C_ENTIDAD, C_PARTIDA, N_PARTIDA,
    C_CAPITULO, N_CAPITULO, C_PROGRAMA, N_PROGRAMA, C_TIPO_CUENTA,
    C_CUENTA, N_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO,
    C_SCN, C_CAGENTE, DEVENGO
INTO #ejec_exp_mgg_preliminar
FROM #ejec_exp_mgg_preliminar_raw
"""
res = work_conn.execute(text(sql_ejec_exp_mgg_preliminar))
_log("CREATE #ejec_exp_mgg_preliminar", res.rowcount)


In [ ]:
# /* Crea tabla con la ejecución total de transacciones MGG, Muestra + Resto */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_mgg_total"))
sql_ejec_mgg_total = """
SELECT
    SUBSECTOR, MONEDA, AÑO, MES, C_ENTIDAD, C_PARTIDA, N_PARTIDA,
    C_CAPITULO, N_CAPITULO, C_PROGRAMA, N_PROGRAMA, C_TIPO_CUENTA,
    C_CUENTA, N_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO,
    C_SCN, C_CAGENTE, DEVENGO
INTO #ejec_mgg_total
FROM #ejec_muestra_mgg
UNION ALL
SELECT
    SUBSECTOR, MONEDA, AÑO, MES, C_ENTIDAD, C_PARTIDA, N_PARTIDA,
    C_CAPITULO, N_CAPITULO, C_PROGRAMA, N_PROGRAMA, C_TIPO_CUENTA,
    C_CUENTA, N_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO,
    C_SCN, C_CAGENTE, DEVENGO
FROM #ejec_exp_mgg_preliminar
"""
res = work_conn.execute(text(sql_ejec_mgg_total))
_log("CREATE #ejec_mgg_total", res.rowcount)


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MGG para COFOG y ANALISIS DESAGREGADO CNF anual */
# /*Desagregado mes*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_e_mgg_aper"))
sql_ejec_e_mgg_aper = """
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    CASE
        WHEN t1.MES IN (1,2,3) THEN 1
        WHEN t1.MES IN (4,5,6) THEN 2
        WHEN t1.MES IN (7,8,9) THEN 3
        ELSE 4
    END AS [TRIM],
    t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
    t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE,
    t3.ACTIVIDAD, t3.NIVEL, t4.C_COFOG_1, t4.N_COFOG_1,
    SUM(t1.DEVENGO) AS DATO
INTO #ejec_e_mgg_aper
FROM #ejec_mgg_total t1
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t3 ON t1.C_ENTIDAD = t3.ENTIDAD
LEFT JOIN GOBGENER.dbo.COFOG t4
    ON t1.C_ENTIDAD = t4.ENTIDAD
   AND t1.C_PARTIDA = t4.C_PARTIDA
   AND t1.C_CAPITULO = t4.C_CAPITULO
   AND t1.C_PROGRAMA = t4.C_PROGRAMA
WHERE t1.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
GROUP BY
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    CASE
        WHEN t1.MES IN (1,2,3) THEN 1
        WHEN t1.MES IN (4,5,6) THEN 2
        WHEN t1.MES IN (7,8,9) THEN 3
        ELSE 4
    END,
    t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
    t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE,
    t3.ACTIVIDAD, t3.NIVEL, t4.C_COFOG_1, t4.N_COFOG_1
"""
res = work_conn.execute(text(sql_ejec_e_mgg_aper))
_log("CREATE #ejec_e_mgg_aper", res.rowcount)


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MGG para COFOG y ANALISIS DESAGREGADO CNF anual */
# /*Agregado año*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_e_mgg_aper_an"))
sql_ejec_e_mgg_aper_an = """
SELECT
    SUBSECTOR, MONEDA, AÑO, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA,
    C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO,
    C_SCN, C_CAGENTE, ACTIVIDAD, NIVEL, C_COFOG_1, N_COFOG_1,
    SUM(DATO) AS DATO
INTO #ejec_e_mgg_aper_an
FROM #ejec_e_mgg_aper
GROUP BY
    SUBSECTOR, MONEDA, AÑO, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA,
    C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO,
    C_SCN, C_CAGENTE, ACTIVIDAD, NIVEL, C_COFOG_1, N_COFOG_1
"""
res = work_conn.execute(text(sql_ejec_e_mgg_aper_an))
_log("CREATE #ejec_e_mgg_aper_an", res.rowcount)


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MGG para CNF anual */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_e_mgg"))
sql_ejec_e_mgg = """
SELECT
    SUBSECTOR, MONEDA, AÑO, [TRIM], MES, C_TIPO_CUENTA, C_CUENTA, C_ITEM,
    C_SCN, C_CAGENTE, ACTIVIDAD, NIVEL,
    SUM(DATO) AS DATO
INTO #ejec_e_mgg
FROM #ejec_e_mgg_aper
GROUP BY
    SUBSECTOR, MONEDA, AÑO, [TRIM], MES, C_TIPO_CUENTA, C_CUENTA, C_ITEM,
    C_SCN, C_CAGENTE, ACTIVIDAD, NIVEL
"""
res = work_conn.execute(text(sql_ejec_e_mgg))
_log("CREATE #ejec_e_mgg", res.rowcount)


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MGG para CNF anual */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_mgg_an"))
sql_ejec_mgg_an = """
SELECT
    SUBSECTOR, MONEDA, AÑO, C_TIPO_CUENTA, C_CUENTA, C_ITEM,
    C_SCN, C_CAGENTE, ACTIVIDAD, NIVEL,
    SUM(DATO) AS DATO
INTO #ejec_mgg_an
FROM #ejec_e_mgg_aper
GROUP BY
    SUBSECTOR, MONEDA, AÑO, C_TIPO_CUENTA, C_CUENTA, C_ITEM,
    C_SCN, C_CAGENTE, ACTIVIDAD, NIVEL
"""
res = work_conn.execute(text(sql_ejec_mgg_an))
_log("CREATE #ejec_mgg_an", res.rowcount)


In [ ]:
# DROP TABLE EJEC_MUESTRA_MGG, EJEC_MUESTRA_MGG_prelim, CRUCE_EJEC_EXP_MGG,
# EJEC_EXP_MGG_prelim, EJEC_EXP_MGG_preliminar, EJEC_EXP_MGG, EJEC_MGG_TOTAL;
# limpia los WORK intermedios al cierre del nodo, igual que el SAS original
# (EJEC_EXP_MGG_prelim y EJEC_EXP_MGG del DROP original nunca se crean en
# este código -> se omiten; #ejec_exp_mgg_preliminar_raw es un artefacto
# propio de esta traducción, también se limpia)
for _t in [
    "#ejec_muestra_mgg",
    "#ejec_muestra_mgg_prelim",
    "#cruce_ejec_exp_mgg",
    "#ejec_exp_mgg_preliminar_raw",
    "#ejec_exp_mgg_preliminar",
    "#ejec_mgg_total",
]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


## APER_CNF_anual

Agrega la ejecución del Gobierno Central y Municipal del año fiscal indicado, aplica reclasificaciones SCN y ajustes puntuales de negocio, e incorpora imputaciones permanentes para consolidar el resultado anual No Financiero (CNF).

*confianza: low · verificador: approve · SAS: PROC SQL CREATE TABLE con múltiples LEFT JOIN, UPDATE/DELETE iterativos, UNION ALL y agregaciones anuales (CNF)*

In [ ]:
# ========= APER_CNF_anual =========
# Adjunta imputación por ajuste en IFE LABORAL 2023 (DATA step SET + IF AÑO=&Year)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion"))
sql_ejecucion = f"""
SELECT * INTO #ejecucion
FROM (
    SELECT * FROM #ejec_cgr_y
    UNION ALL
    SELECT * FROM GOBGENER.dbo.AJUSTE_IFE_LABORAL
) base
WHERE AÑO = {int(Year)}
"""
res = work_conn.execute(text(sql_ejecucion))
_log("CREATE #ejecucion", res.rowcount)


In [ ]:
# Analiticos predefinidos
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_aper_an"))
sql_a_prelim = """
SELECT MIN(t1.SUBSECTOR) AS SUBSECTOR,
       MIN(t1.MONEDA) AS MONEDA,
       MIN(t1.AÑO) AS AÑO,
       CASE WHEN t1.MES IN (1,2,3) THEN 1
            WHEN t1.MES IN (4,5,6) THEN 2
            WHEN t1.MES IN (7,8,9) THEN 3
            ELSE 4 END AS TRIM,
       t1.MES,
       t1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       t4.ACTIVIDAD,
       t4.NIVEL,
       SUM(t1.DEVENG * t3.DATO / 1000000) AS DEVENGO
INTO #ejecucion_a_prelim_aper_an
FROM #ejecucion t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
    ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
    ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t1.C_CUENTA NOT IN ('05','13','24','33') AND t1.C_ENTIDAD NOT IN (5601,10201)
GROUP BY t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA,
         t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t2.C_SCN,
         t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
         t4.ACTIVIDAD, t4.NIVEL
"""
res = work_conn.execute(text(sql_a_prelim))
_log("CREATE #ejecucion_a_prelim_aper_an", res.rowcount)


In [ ]:
# Analiticos predefinidos: cuenta 32 con analíticos 04001000/04002000 se tratan aparte (ver EJECUCION_B)
work_conn.execute(text(
    "DELETE FROM #ejecucion_a_prelim_aper_an WHERE C_CUENTA='32' AND C_ANALITICO IN ('04001000','04002000')"
))

# Analiticos predefinidos: AJUSTE COYUNTURA - se elimina de indicador transacción CORFO
# que se desagrega anualmente entre ventas y D45
res = work_conn.execute(text(
    "DELETE FROM #ejecucion_a_prelim_aper_an "
    "WHERE C_ENTIDAD=3801 AND C_CUENTA='06' AND C_ITEM='01' AND C_ASIGNACION='000' AND C_ANALITICO IN ('01000000')"
))
_log("DELETE #ejecucion_a_prelim_aper_an (cuenta 32 / ajuste CORFO)", res.rowcount)


In [ ]:
# Analiticos predefinidos: reasignación D39 2% constitucional 2008 para Transantiago
work_conn.execute(text(
    "UPDATE #ejecucion_a_prelim_aper_an SET C_CAGENTE='S11' "
    "WHERE C_ENTIDAD=19703 AND C_CUENTA='26' AND C_ITEM='03' AND C_ASIGNACION='000' "
    "AND C_ANALITICO IN ('03000000') AND MONEDA='P' AND AÑO=2008 AND MES IN (10,11,12)"
))
work_conn.execute(text(
    "UPDATE #ejecucion_a_prelim_aper_an SET C_SCN='D39' "
    "WHERE C_ENTIDAD=19703 AND C_CUENTA='26' AND C_ITEM='03' AND C_ASIGNACION='000' "
    "AND C_ANALITICO IN ('03000000') AND MONEDA='P' AND AÑO=2008 AND MES IN (10,11,12)"
))
res = work_conn.execute(text(
    "UPDATE #ejecucion_a_prelim_aper_an SET C_TRANSACCION='4139' "
    "WHERE C_ENTIDAD=19703 AND C_CUENTA='26' AND C_ITEM='03' AND C_ASIGNACION='000' "
    "AND C_ANALITICO IN ('03000000') AND MONEDA='P' AND AÑO=2008 AND MES IN (10,11,12)"
))
_log("UPDATE #ejecucion_a_prelim_aper_an (Transantiago D39 2008)", res.rowcount)

# Analiticos predefinidos: AJUSTE COBRE BRUTO (Financiamiento Capacidades Estratégicas
# Defensa Nacional) - reasignación entre D75 y D42
work_conn.execute(text(
    "UPDATE #ejecucion_a_prelim_aper_an SET C_SCN='D42' "
    "WHERE C_ENTIDAD=19701 AND C_CUENTA='08' AND C_ITEM='99' AND C_ASIGNACION='999' "
    "AND SUBSTRING(C_ANALITICO,1,5)='99999' AND MONEDA='D' AND AÑO>2019"
))
work_conn.execute(text(
    "UPDATE #ejecucion_a_prelim_aper_an SET C_TRANSACCION='42222' "
    "WHERE C_ENTIDAD=19701 AND C_CUENTA='08' AND C_ITEM='99' AND C_ASIGNACION='999' "
    "AND SUBSTRING(C_ANALITICO,1,5)='99999' AND MONEDA='D' AND AÑO>2019"
))
res = work_conn.execute(text(
    "UPDATE #ejecucion_a_prelim_aper_an SET N_CUENTA='Cobre bruto' "
    "WHERE C_ENTIDAD=19701 AND C_CUENTA='08' AND C_ITEM='99' AND C_ASIGNACION='999' "
    "AND SUBSTRING(C_ANALITICO,1,5)='99999' AND MONEDA='D' AND AÑO>2019"
))
_log("UPDATE #ejecucion_a_prelim_aper_an (Cobre bruto D75->D42)", res.rowcount)


In [ ]:
# Analiticos predefinidos (sigue): excluye transacciones fuera de alcance CNF
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_aper_an"))
sql_a = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
       t1.ACTIVIDAD, t1.NIVEL, SUM(t1.DEVENGO) AS Dato
INTO #ejecucion_a_aper_an
FROM #ejecucion_a_prelim_aper_an t1
WHERE t1.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
   OR t1.C_SCN IS NULL
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
         t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
         t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
         t1.ACTIVIDAD, t1.NIVEL
"""
res = work_conn.execute(text(sql_a))
_log("CREATE #ejecucion_a_aper_an", res.rowcount)


In [ ]:
# Analiticos fijos cuenta 32
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_b_aper_an"))
sql_b = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
       t4.ACTIVIDAD, t4.NIVEL, SUM(t1.DEVENG * t3.DATO / 1000000) AS Dato
INTO #ejecucion_b_aper_an
FROM #ejecucion t1
LEFT JOIN GOBGENER.dbo.T_SCN_1 t2
    ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
   AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
    ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t1.C_CUENTA='32'
  AND (t2.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9') OR t2.C_SCN IS NULL)
  AND t1.C_ANALITICO IN ('04001000','04002000')
  AND t1.C_ENTIDAD NOT IN (5601,10201)
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
         t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
         t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
         t4.ACTIVIDAD, t4.NIVEL
"""
res = work_conn.execute(text(sql_b))
_log("CREATE #ejecucion_b_aper_an", res.rowcount)


In [ ]:
# Analíticos variables
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_c_aper_an_prelim"))
sql_c_prelim = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
       t4.ACTIVIDAD, t4.NIVEL, SUM(t1.DEVENG * t3.DATO / 1000000) AS Dato
INTO #ejecucion_c_aper_an_prelim
FROM #ejecucion t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
   AND t1.C_ENTIDAD = t2.ENTIDAD AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA
   AND t1.C_ITEM = t2.C_ITEM AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
    ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t1.C_CUENTA IN ('05','13','24','33')
  AND (t2.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9','') OR t2.C_SCN IS NULL)
  AND (t2.OBS NOT IN ('CR18_difcoy','CR18_difact') OR t2.OBS IS NULL)
  AND t1.C_ENTIDAD NOT IN (5601,10201)
  AND (t2.C_SCN NOT IN ('TC') OR t2.C_SCN IS NULL)
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
         t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
         t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
         t4.ACTIVIDAD, t4.NIVEL
"""
res = work_conn.execute(text(sql_c_prelim))
_log("CREATE #ejecucion_c_aper_an_prelim", res.rowcount)


In [ ]:
# Analiticos variables: reasignación contragente ayudas IPS (Bono Ayuda Familiar 18O, Bono Covid)
work_conn.execute(text(
    "UPDATE #ejecucion_c_aper_an_prelim SET C_CAGENTE='S14' "
    "WHERE C_ENTIDAD=24101 AND C_CUENTA='24' AND C_ITEM='01' AND C_ASIGNACION='006' "
    "AND C_ANALITICO IN ('01006000') AND MONEDA='P' AND AÑO=2019 AND MES=12"
))
work_conn.execute(text(
    "UPDATE #ejecucion_c_aper_an_prelim SET C_CAGENTE='S14' "
    "WHERE C_ENTIDAD=24101 AND C_CUENTA='24' AND C_ITEM='01' AND C_ASIGNACION='006' "
    "AND C_ANALITICO IN ('01006000') AND MONEDA='P' AND AÑO=2020 AND MES=4"
))
# Reasignación nivel de gobierno para Programa Ampliado de Inmunizaciones
work_conn.execute(text(
    "UPDATE #ejecucion_c_aper_an_prelim SET NIVEL='MUNICIPAL' "
    "WHERE C_ENTIDAD IN (11105,22701,22750) AND C_CUENTA='24' AND C_ITEM='01' AND C_ASIGNACION='004'"
))
# Reasignación nivel de D63 municipal
work_conn.execute(text(
    "UPDATE #ejecucion_c_aper_an_prelim SET NIVEL='MUNICIPAL' "
    "WHERE NIVEL='REGIONAL' AND C_SCN IN ('D63','D63-P2','D63-P') AND C_ITEM='03' "
    "AND C_ASIGNACION IN ('001','003','006','011','013','015','016','017','018','020','021','100','101')"
))
# Beca Especial Carbón (previo 2020)
work_conn.execute(text(
    "UPDATE #ejecucion_c_aper_an_prelim SET C_CAGENTE='S14' "
    "WHERE C_ENTIDAD=6403 AND C_CUENTA='24' AND C_ITEM='01' AND C_ASIGNACION='270' "
    "AND C_ANALITICO IN ('01270000') AND MONEDA='P' AND AÑO<2020"
))
work_conn.execute(text(
    "UPDATE #ejecucion_c_aper_an_prelim SET C_SCN='D75' "
    "WHERE C_ENTIDAD=6403 AND C_CUENTA='24' AND C_ITEM='01' AND C_ASIGNACION='270' "
    "AND C_ANALITICO IN ('01270000') AND MONEDA='P' AND AÑO<2020"
))
res = work_conn.execute(text(
    "UPDATE #ejecucion_c_aper_an_prelim SET C_TRANSACCION='4345' "
    "WHERE C_ENTIDAD=6403 AND C_CUENTA='24' AND C_ITEM='01' AND C_ASIGNACION='270' "
    "AND C_ANALITICO IN ('01270000') AND MONEDA='P' AND AÑO<2020"
))
_log("UPDATE #ejecucion_c_aper_an_prelim (ajustes IPS/PAI/D63/Beca Carbón)", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_c_aper_an"))
sql_c = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
       t1.ACTIVIDAD, t1.NIVEL, SUM(t1.Dato) AS Dato
INTO #ejecucion_c_aper_an
FROM #ejecucion_c_aper_an_prelim t1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
         t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
         t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
         t1.ACTIVIDAD, t1.NIVEL
"""
res = work_conn.execute(text(sql_c))
_log("CREATE #ejecucion_c_aper_an", res.rowcount)


In [ ]:
# Transacciones diferenciadas de coyuntura, con clasificación anual
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_d_aper_an_prelim"))
sql_d_prelim = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       CASE WHEN t1.MES IN (1,2,3) THEN 1
            WHEN t1.MES IN (4,5,6) THEN 2
            WHEN t1.MES IN (7,8,9) THEN 3
            ELSE 4 END AS TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
       t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t2.C_SCN, t2.C_CAGENTE,
       t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP, t4.ACTIVIDAD, t4.NIVEL,
       (t1.DEVENG * t3.DATO / 1000000) AS Dato
INTO #ejecucion_d_aper_an_prelim
FROM #ejecucion t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
   AND t1.C_ENTIDAD = t2.ENTIDAD AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA
   AND t1.C_ITEM = t2.C_ITEM AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
    ON t1.C_ENTIDAD = t4.ENTIDAD
LEFT JOIN GOBGENER.dbo.COFOG t5
    ON t1.C_ENTIDAD = t5.ENTIDAD AND t1.C_PARTIDA = t5.C_PARTIDA AND t1.C_CAPITULO = t5.C_CAPITULO
   AND t1.C_PROGRAMA = t5.C_PROGRAMA
WHERE t2.OBS IN ('CR18_difcoy') AND t1.C_ENTIDAD NOT IN (5601,10201) AND t2.C_SCN NOT IN ('TC')
  AND t2.C_ACTIVO NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
"""
res = work_conn.execute(text(sql_d_prelim))
_log("CREATE #ejecucion_d_aper_an_prelim", res.rowcount)

# Reasignación nivel de D63 municipal
work_conn.execute(text(
    "UPDATE #ejecucion_d_aper_an_prelim SET NIVEL='MUNICIPAL' "
    "WHERE NIVEL='REGIONAL' AND C_SCN IN ('D63','D63-P2','D63-P') AND C_ITEM='03' "
    "AND C_ASIGNACION IN ('001','003','006','011','013','015','016','017','018','020','021','100','101')"
))


In [ ]:
# Glosas comunes GORE: separa D63-P2/D63-P regional y municipal para aperturar por COFOG
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_da_aper_an_prelim"))
sql_da_prelim = """
SELECT t1.*
INTO #ejecucion_da_aper_an_prelim
FROM #ejecucion_d_aper_an_prelim t1
WHERE t1.C_SCN IN ('D63-P2','D63-P') AND t1.NIVEL IN ('REGIONAL','MUNICIPAL')
"""
work_conn.execute(text(sql_da_prelim))

res = work_conn.execute(text(
    "DELETE FROM #ejecucion_d_aper_an_prelim WHERE C_SCN IN ('D63-P2','D63-P') AND NIVEL IN ('REGIONAL','MUNICIPAL')"
))
_log("DELETE #ejecucion_d_aper_an_prelim (D63-P2/D63-P movido a apertura COFOG)", res.rowcount)


In [ ]:
# Realiza apertura de glosas COFOG de glosas comunes de GOREs
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_da_aper_an"))
sql_da = """
SELECT t2.SUBSECTOR, t2.MONEDA, t2.AÑO, t2.TRIM, t2.MES, t2.C_ENTIDAD, t2.C_PARTIDA, t2.C_CAPITULO,
       t2.C_PROGRAMA, t2.C_TIPO_CUENTA, t2.C_CUENTA, t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO,
       t2.N_ANALITICO, t1.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP, t2.ACTIVIDAD, t2.NIVEL, SUM(t2.Dato * t1.POND) AS Dato
INTO #ejecucion_da_aper_an
FROM GOBGENER.dbo.EST_GLOSASCOM t1
LEFT JOIN #ejecucion_da_aper_an_prelim t2
    ON t2.AÑO = t1.AÑO AND t1.C_ENTIDAD = t2.C_ENTIDAD AND t1.C_ITEM = t2.C_ITEM
WHERE t2.NIVEL IS NOT NULL
GROUP BY t2.SUBSECTOR, t2.MONEDA, t2.AÑO, t2.TRIM, t2.MES, t2.C_ENTIDAD, t2.C_PARTIDA, t2.C_CAPITULO,
         t2.C_PROGRAMA, t2.C_TIPO_CUENTA, t2.C_CUENTA, t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO,
         t2.N_ANALITICO, t1.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
         t2.C_ACTIVO, t2.C_CUP, t2.ACTIVIDAD, t2.NIVEL
"""
work_conn.execute(text(sql_da))

# Une tablas proceso D (DATA step SET): acumula la apertura COFOG en D_prelim
res = work_conn.execute(text(
    "INSERT INTO #ejecucion_d_aper_an_prelim "
    "(SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, C_TIPO_CUENTA, "
    "C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, C_SCN, C_CAGENTE, C_TRANSACCION, "
    "T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, ACTIVIDAD, NIVEL, Dato) "
    "SELECT SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, C_TIPO_CUENTA, "
    "C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, C_SCN, C_CAGENTE, C_TRANSACCION, "
    "T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, ACTIVIDAD, NIVEL, Dato FROM #ejecucion_da_aper_an"
))
_log("APPEND #ejecucion_d_aper_an_prelim (apertura COFOG glosas comunes)", res.rowcount)


In [ ]:
# Resumen anual de proceso D
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_d_aper_an"))
sql_d = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
       t1.ACTIVIDAD, t1.NIVEL, SUM(t1.Dato) AS Dato
INTO #ejecucion_d_aper_an
FROM #ejecucion_d_aper_an_prelim t1
WHERE t1.NIVEL IS NOT NULL
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
         t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
         t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
         t1.ACTIVIDAD, t1.NIVEL
"""
res = work_conn.execute(text(sql_d))
_log("CREATE #ejecucion_d_aper_an", res.rowcount)


In [ ]:
# Transacciones diferenciadas por actividad
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_f_aper_an"))
sql_f = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
       t2.C_CUP AS ACTIVIDAD, t4.NIVEL, SUM(t1.DEVENG * t3.DATO / 1000000) AS Dato
INTO #ejecucion_f_aper_an
FROM #ejecucion t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
   AND t1.C_ENTIDAD = t2.ENTIDAD AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA
   AND t1.C_ITEM = t2.C_ITEM AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
    ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t2.OBS IN ('CR18_difact') AND t1.C_ENTIDAD NOT IN (5601,10201) AND t2.C_SCN NOT IN ('TC')
  AND t2.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
         t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
         t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
         t4.NIVEL
"""
res = work_conn.execute(text(sql_f))
_log("CREATE #ejecucion_f_aper_an", res.rowcount)


In [ ]:
# Une tablas de transferencias consolidables detallado anual (DROP=C_COFOG_1 N_COFOG_1 del original;
# se fuerza C_ACTIVO='TC' sobre las 4 fuentes dejadas por los nodos de TCons Mixtas_M/MGCG/MM/MGG)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_e_aper_an"))
sql_e = """
SELECT SUBSECTOR, MONEDA, AÑO, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, C_TIPO_CUENTA, C_CUENTA,
       C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, C_SCN, C_CAGENTE, C_TRANSACCION, T_TRANSACCION,
       N_CUENTA, 'TC' AS C_ACTIVO, C_CUP, ACTIVIDAD, NIVEL, Dato
INTO #ejecucion_e_aper_an
FROM (
    SELECT SUBSECTOR, MONEDA, AÑO, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, C_TIPO_CUENTA, C_CUENTA,
           C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, C_SCN, C_CAGENTE, C_TRANSACCION, T_TRANSACCION,
           N_CUENTA, C_CUP, ACTIVIDAD, NIVEL, Dato FROM #ejec_e_mixt_aper_an
    UNION ALL
    SELECT SUBSECTOR, MONEDA, AÑO, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, C_TIPO_CUENTA, C_CUENTA,
           C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, C_SCN, C_CAGENTE, C_TRANSACCION, T_TRANSACCION,
           N_CUENTA, C_CUP, ACTIVIDAD, NIVEL, Dato FROM #ejec_e_mgcg_aper_an
    UNION ALL
    SELECT SUBSECTOR, MONEDA, AÑO, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, C_TIPO_CUENTA, C_CUENTA,
           C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, C_SCN, C_CAGENTE, C_TRANSACCION, T_TRANSACCION,
           N_CUENTA, C_CUP, ACTIVIDAD, NIVEL, Dato FROM #ejec_e_mm_aper_an
    UNION ALL
    SELECT SUBSECTOR, MONEDA, AÑO, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, C_TIPO_CUENTA, C_CUENTA,
           C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, C_SCN, C_CAGENTE, C_TRANSACCION, T_TRANSACCION,
           N_CUENTA, C_CUP, ACTIVIDAD, NIVEL, Dato FROM #ejec_e_mgg_aper_an
) u
"""
res = work_conn.execute(text(sql_e))
_log("CREATE #ejecucion_e_aper_an", res.rowcount)


In [ ]:
# Une tablas salida CNF anual (Gobierno Central)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_ext_cgr_aper_prev_an"))
sql_ext = """
SELECT * INTO #ejecucion_ext_cgr_aper_prev_an
FROM (
    SELECT * FROM #ejecucion_a_aper_an
    UNION ALL
    SELECT * FROM #ejecucion_b_aper_an
    UNION ALL
    SELECT * FROM #ejecucion_c_aper_an
    UNION ALL
    SELECT * FROM #ejecucion_d_aper_an
    UNION ALL
    SELECT * FROM #ejecucion_e_aper_an
    UNION ALL
    SELECT * FROM #ejecucion_f_aper_an
) u
"""
res = work_conn.execute(text(sql_ext))
_log("CREATE #ejecucion_ext_cgr_aper_prev_an", res.rowcount)

# Limpieza de tablas intermedias ya consolidadas
for _t in ["#ejecucion_a_prelim_aper_an", "#ejecucion_a_aper_an", "#ejecucion_b_aper_an",
           "#ejecucion_c_aper_an", "#ejecucion_c_aper_an_prelim", "#ejecucion_d_aper_an",
           "#ejecucion_d_aper_an_prelim", "#ejecucion_da_aper_an", "#ejecucion_da_aper_an_prelim",
           "#ejecucion_e_aper_an", "#ejecucion_e_aper_prev_an", "#ejecucion_f_aper_an",
           "#ejecucion_ext_dip_aper_prev_an"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


In [ ]:
# Incorporación de imputaciones permanentes: mapeo actividad/nivel por partida-capitulo-programa
work_conn.execute(text("DROP TABLE IF EXISTS #cr18_apertura_pacapr"))
sql_pacapr = """
SELECT DISTINCT t1.ACTIVIDAD, t2.C_PARTIDA, t2.C_CAPITULO, t2.C_PROGRAMA, t1.NIVEL
INTO #cr18_apertura_pacapr
FROM GOBGENER.dbo.CR18_APERTURA t1
LEFT JOIN GOBGENER.dbo.ENTIDADES t2 ON t1.ENTIDAD = t2.C_ENTIDAD
"""
work_conn.execute(text(sql_pacapr))

work_conn.execute(text("DROP TABLE IF EXISTS #ejec_ext_dip_aper_imput_perm_an"))
sql_imput_perm = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
       t2.ACTIVIDAD, t2.NIVEL, SUM(t1.Dato) AS DATO
INTO #ejec_ext_dip_aper_imput_perm_an
FROM GOBGENER.dbo.IMPUT_GC_DIP_APER t1
LEFT JOIN #cr18_apertura_pacapr t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
WHERE t1.AÑO = {int(Year)}
  AND t1.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
         t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
         t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
         t2.ACTIVIDAD, t2.NIVEL
"""
res = work_conn.execute(text(sql_imput_perm))
_log("CREATE #ejec_ext_dip_aper_imput_perm_an", res.rowcount)


In [ ]:
# Une imputaciones permanentes a la salida CNF anual
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cnf_cgr_aper_an"))
sql_cnf_cgr = """
SELECT SUBSECTOR, MONEDA, AÑO, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, C_TIPO_CUENTA, C_CUENTA,
       C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, C_SCN, C_CAGENTE, C_TRANSACCION, T_TRANSACCION,
       N_CUENTA, C_ACTIVO, C_CUP, ACTIVIDAD, NIVEL, Dato
INTO #ejecucion_cnf_cgr_aper_an
FROM (
    SELECT SUBSECTOR, MONEDA, AÑO, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, C_TIPO_CUENTA, C_CUENTA,
           C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, C_SCN, C_CAGENTE, C_TRANSACCION, T_TRANSACCION,
           N_CUENTA, C_ACTIVO, C_CUP, ACTIVIDAD, NIVEL, Dato FROM #ejecucion_ext_cgr_aper_prev_an
    UNION ALL
    SELECT SUBSECTOR, MONEDA, AÑO, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, C_TIPO_CUENTA, C_CUENTA,
           C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, C_SCN, C_CAGENTE, C_TRANSACCION, T_TRANSACCION,
           N_CUENTA, C_ACTIVO, C_CUP, ACTIVIDAD, NIVEL, DATO AS Dato FROM #ejec_ext_dip_aper_imput_perm_an
) u
"""
res = work_conn.execute(text(sql_cnf_cgr))
_log("CREATE #ejecucion_cnf_cgr_aper_an", res.rowcount)

# Reasignación de nivel regional a central, por metodología de acuerdo a características actuales de GOREs
# (la reasignación P3->D73 que precedía a esta en el SAS queda comentada /* ... */ en el original: no se traduce)
work_conn.execute(text("UPDATE #ejecucion_cnf_cgr_aper_an SET NIVEL='CENTRAL' WHERE NIVEL='REGIONAL'"))

# Reasignación FBCF realizada por municipalidades pero medida en ejecución GC
work_conn.execute(text(
    "UPDATE #ejecucion_cnf_cgr_aper_an SET NIVEL='MUNICIPAL' "
    "WHERE NIVEL='CENTRAL' AND C_SCN='P51' AND C_CAGENTE='S1313'"
))

# Eliminación de transacciones redundantes en consolidación
for _scn, _nivel, _cagente in [
    ("D73", "CENTRAL", "S13"), ("D93", "CENTRAL", "S13"),
    ("D73", "CENTRAL", "S1312"), ("D93", "CENTRAL", "S1312"),
    ("D73", "CENTRAL", "S131"), ("D93", "CENTRAL", "S131"),
    ("D73", "PREVISION", "S1314"), ("D93", "PREVISION", "S1314"),
]:
    work_conn.execute(text(
        f"DELETE FROM #ejecucion_cnf_cgr_aper_an WHERE C_SCN='{_scn}' AND NIVEL='{_nivel}' AND C_CAGENTE='{_cagente}'"
    ))

work_conn.execute(text("DROP TABLE IF EXISTS #ejec_ext_dip_aper_imput_perm_an"))


In [ ]:
# EJECUCION MUNICIPAL: salida predefinidos (histórico, antes de 2022)
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cnf_mun_a_an_aper"))
sql_mun_a = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, 3 AS C_ENTIDAD, 'MUNI' AS C_PARTIDA, 'MUNI' AS C_CAPITULO,
       'MUNI' AS C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
       SUBSTRING(t1.C_ANALITICO,1,2) AS C_ITEM, SUBSTRING(t1.C_ANALITICO,3,3) AS C_ASIGNACION,
       t1.C_ANALITICO, t2.ANALITICO AS N_ANALITICO, t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION,
       t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
       CASE WHEN t1.C_T_AREA='GM' THEN 'AAPP' WHEN t1.C_T_AREA='ED' THEN 'EEPP'
            WHEN t1.C_T_AREA='SL' THEN 'SSPP' END AS ACTIVIDAD,
       'MUNICIPAL' AS NIVEL,
       SUM(t1.DEVENG / 1000000) AS Dato
INTO #ejec_cnf_mun_a_an_aper
FROM GOBGENER.dbo.EJECUCION_MUN t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1_MUN t2
    ON t1.C_TIPO_CUENTA = t2.C_TIPO_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
WHERE t1.C_CUENTA NOT IN ('05','13','24','33')
  AND t1.AÑO = {int(Year)} AND t1.AÑO < 2022 AND t1.C_T_AREA NOT IN ('CM')
  AND (t2.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9') OR t2.C_SCN IS NULL)
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA,
         SUBSTRING(t1.C_ANALITICO,1,2), SUBSTRING(t1.C_ANALITICO,3,3), t1.C_ANALITICO, t2.ANALITICO,
         t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
         CASE WHEN t1.C_T_AREA='GM' THEN 'AAPP' WHEN t1.C_T_AREA='ED' THEN 'EEPP'
              WHEN t1.C_T_AREA='SL' THEN 'SSPP' END
"""
work_conn.execute(text(sql_mun_a))

# EJECUCION MUNICIPAL: salida variables (histórico, antes de 2022)
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cnf_mun_b_an_aper"))
sql_mun_b = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, 3 AS C_ENTIDAD, 'MUNI' AS C_PARTIDA, 'MUNI' AS C_CAPITULO,
       'MUNI' AS C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
       SUBSTRING(t1.C_ANALITICO,1,2) AS C_ITEM, SUBSTRING(t1.C_ANALITICO,3,3) AS C_ASIGNACION,
       t1.C_ANALITICO, t1.ANALITICO AS N_ANALITICO, t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION,
       t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
       CASE WHEN t1.C_T_AREA='GM' THEN 'AAPP' WHEN t1.C_T_AREA='ED' THEN 'EEPP'
            WHEN t1.C_T_AREA='SL' THEN 'SSPP' END AS ACTIVIDAD,
       'MUNICIPAL' AS NIVEL,
       SUM(t1.DEVENG / 1000000) AS Dato
INTO #ejec_cnf_mun_b_an_aper
FROM GOBGENER.dbo.EJECUCION_MUN t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2_MUN t2
    ON t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_T_AREA = t2.C_T_AREA
WHERE t1.C_CUENTA IN ('05','13','24','33')
  AND t1.AÑO = {int(Year)} AND t1.AÑO < 2022 AND t1.C_T_AREA NOT IN ('CM')
  AND (t2.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9') OR t2.C_SCN IS NULL)
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA,
         SUBSTRING(t1.C_ANALITICO,1,2), SUBSTRING(t1.C_ANALITICO,3,3), t1.C_ANALITICO, t1.ANALITICO,
         t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
         CASE WHEN t1.C_T_AREA='GM' THEN 'AAPP' WHEN t1.C_T_AREA='ED' THEN 'EEPP'
              WHEN t1.C_T_AREA='SL' THEN 'SSPP' END
"""
res = work_conn.execute(text(sql_mun_b))
_log("CREATE #ejec_cnf_mun_a_an_aper / #ejec_cnf_mun_b_an_aper", res.rowcount)


In [ ]:
# Ejecución municipal nueva (desde 2022 en adelante): predefinidos
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cnf_mun_a_an_nuevo"))
sql_mun_a_nuevo = f"""
SELECT SUBSTRING(t1.SUBSECTOR,2,1) AS SUBSECTOR, SUBSTRING(t1.MONEDA,3,1) AS MONEDA, t1.AÑO,
       3 AS C_ENTIDAD, 'MUNI' AS C_PARTIDA, 'MUNI' AS C_CAPITULO, 'MUNI' AS C_PROGRAMA,
       CASE WHEN t1.C_T_CUENTA = 1 THEN 'Y' WHEN t1.C_T_CUENTA = 2 THEN 'G' ELSE '' END AS C_TIPO_CUENTA,
       t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t2.N_ASIGNACION AS N_ANALITICO,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
       CASE WHEN t1.C_SERVICIO=1 THEN 'AAPP' WHEN t1.C_SERVICIO=2 THEN 'EEPP'
            WHEN t1.C_SERVICIO=3 THEN 'SSPP' END AS ACTIVIDAD,
       'MUNICIPAL' AS NIVEL,
       SUM(t1.DEVENGO / 1000000) AS Dato
INTO #ejec_cnf_mun_a_an_nuevo
FROM GOBGENER.dbo.EJECUCION_MUN_P t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1_MUN_P t2
    ON t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO_2 = t2.C_ANALITICO_2
WHERE t1.C_CUENTA NOT IN ('05','13','24','33')
  AND (CASE WHEN t1.C_T_CUENTA = 1 THEN 'Y' WHEN t1.C_T_CUENTA = 2 THEN 'G' ELSE '' END) = t2.C_TIPO_CUENTA
  AND t1.AÑO = {int(Year)} AND t1.AÑO >= 2022 AND t1.C_SERVICIO NOT IN (4)
  AND (t2.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9') OR t2.C_SCN IS NULL)
GROUP BY SUBSTRING(t1.SUBSECTOR,2,1), SUBSTRING(t1.MONEDA,3,1), t1.AÑO, t1.C_T_CUENTA, t1.C_CUENTA,
         t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t2.N_ASIGNACION, t2.C_SCN, t2.C_CAGENTE,
         t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
         CASE WHEN t1.C_SERVICIO=1 THEN 'AAPP' WHEN t1.C_SERVICIO=2 THEN 'EEPP'
              WHEN t1.C_SERVICIO=3 THEN 'SSPP' END
"""
work_conn.execute(text(sql_mun_a_nuevo))

# Ejecución municipal nueva (desde 2022 en adelante): variables
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cnf_mun_b_an_nuevo"))
sql_mun_b_nuevo = f"""
SELECT SUBSTRING(t1.SUBSECTOR,2,1) AS SUBSECTOR, SUBSTRING(t1.MONEDA,3,1) AS MONEDA, t1.AÑO,
       3 AS C_ENTIDAD, 'MUNI' AS C_PARTIDA, 'MUNI' AS C_CAPITULO, 'MUNI' AS C_PROGRAMA,
       CASE WHEN t1.C_T_CUENTA = 1 THEN 'Y' WHEN t1.C_T_CUENTA = 2 THEN 'G' ELSE '' END AS C_TIPO_CUENTA,
       t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t2.N_ASIGNACION AS N_ANALITICO,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
       CASE WHEN t1.C_SERVICIO=1 THEN 'AAPP' WHEN t1.C_SERVICIO=2 THEN 'EEPP'
            WHEN t1.C_SERVICIO=3 THEN 'SSPP' END AS ACTIVIDAD,
       'MUNICIPAL' AS NIVEL,
       SUM(t1.DEVENGO / 1000000) AS Dato
INTO #ejec_cnf_mun_b_an_nuevo
FROM GOBGENER.dbo.EJECUCION_MUN_P t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2_MUN_P t2
    ON t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_SERVICIO = t2.C_SERVICIO
WHERE t1.C_CUENTA IN ('05','13','24','33')
  AND t1.AÑO = {int(Year)} AND t1.AÑO >= 2022 AND t1.C_SERVICIO NOT IN (4)
  AND (t2.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9') OR t2.C_SCN IS NULL)
GROUP BY SUBSTRING(t1.SUBSECTOR,2,1), SUBSTRING(t1.MONEDA,3,1), t1.AÑO, t1.C_T_CUENTA, t1.C_CUENTA,
         t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t2.N_ASIGNACION, t2.C_SCN, t2.C_CAGENTE,
         t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
         CASE WHEN t1.C_SERVICIO=1 THEN 'AAPP' WHEN t1.C_SERVICIO=2 THEN 'EEPP'
              WHEN t1.C_SERVICIO=3 THEN 'SSPP' END
"""
res = work_conn.execute(text(sql_mun_b_nuevo))
_log("CREATE #ejec_cnf_mun_a_an_nuevo / #ejec_cnf_mun_b_an_nuevo", res.rowcount)


In [ ]:
# Unión de ambos tipos de ejecución municipal (predefinidos/variables, histórico/nuevo)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cnf_mun_an_aper"))
sql_mun_union = """
SELECT * INTO #ejecucion_cnf_mun_an_aper
FROM (
    SELECT * FROM #ejec_cnf_mun_a_an_aper
    UNION ALL
    SELECT * FROM #ejec_cnf_mun_b_an_aper
    UNION ALL
    SELECT * FROM #ejec_cnf_mun_a_an_nuevo
    UNION ALL
    SELECT * FROM #ejec_cnf_mun_b_an_nuevo
) u
"""
res = work_conn.execute(text(sql_mun_union))
_log("CREATE #ejecucion_cnf_mun_an_aper", res.rowcount)

# Reasignación de P3 para cuentas de subsector
work_conn.execute(text("UPDATE #ejecucion_cnf_mun_an_aper SET C_SCN='D73' WHERE C_SCN='P3'"))

# Eliminación de transacciones redundantes en consolidación
work_conn.execute(text("DELETE FROM #ejecucion_cnf_mun_an_aper WHERE C_SCN='D73' AND C_CAGENTE='S1313'"))
work_conn.execute(text("DELETE FROM #ejecucion_cnf_mun_an_aper WHERE C_SCN='D93' AND C_CAGENTE='S1313'"))

# Eliminación de D63 por ya estar estimadas desde el origen (EDO), Gobierno Central
work_conn.execute(text("DELETE FROM #ejecucion_cnf_mun_an_aper WHERE C_SCN='D63'"))


In [ ]:
# Unión de ambos tipos de ejecución, Gobierno Central + Municipal
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cnf_an_aper"))
sql_cnf_an_aper = """
SELECT * INTO #ejecucion_cnf_an_aper
FROM (
    SELECT * FROM #ejecucion_cnf_cgr_aper_an
    UNION ALL
    SELECT * FROM #ejecucion_cnf_mun_an_aper
) u
"""
res = work_conn.execute(text(sql_cnf_an_aper))
_log("CREATE #ejecucion_cnf_an_aper", res.rowcount)

# Se eliminan dividendos de CORFO porque se tomarán los del sectorialista de dividendos
work_conn.execute(text("DELETE FROM #ejecucion_cnf_an_aper WHERE C_SCN='D421' AND ACTIVIDAD='CORFO'"))
# Se eliminan subvenciones porque se toman las calculadas aparte
work_conn.execute(text("DELETE FROM #ejecucion_cnf_an_aper WHERE C_SCN='D39'"))
# Se elimina D41 dólar, porque se toma dado de contragente S2 de Balanza de Pagos
work_conn.execute(text(
    "DELETE FROM #ejecucion_cnf_an_aper WHERE C_SCN='D41' AND C_TIPO_CUENTA='Y' AND MONEDA='D'"
))
# Se eliminan datos de producción dado que se ocupan los publicados
work_conn.execute(text("DELETE FROM #ejecucion_cnf_an_aper WHERE C_SCN IN ('D11','D12','P2')"))

# Estandarización de salida para cálculo de cuenta
work_conn.execute(text("UPDATE #ejecucion_cnf_an_aper SET C_SCN='D11' WHERE C_SCN='D12'"))
work_conn.execute(text("UPDATE #ejecucion_cnf_an_aper SET C_SCN='D61' WHERE C_SCN='D6111'"))
work_conn.execute(text("UPDATE #ejecucion_cnf_an_aper SET C_SCN='D75' WHERE C_SCN='D74'"))
work_conn.execute(text("UPDATE #ejecucion_cnf_an_aper SET C_SCN='D99' WHERE C_SCN IN ('D91','D92','D991')"))
work_conn.execute(text("UPDATE #ejecucion_cnf_an_aper SET C_SCN='K2' WHERE C_SCN='P513'"))
work_conn.execute(text("UPDATE #ejecucion_cnf_an_aper SET C_SCN='P51' WHERE C_SCN='P511'"))
work_conn.execute(text("UPDATE #ejecucion_cnf_an_aper SET C_SCN='D42' WHERE C_SCN='D421'"))

for _t in ["#ejec_cnf_mun_a_an_aper_prelim", "#ejec_cnf_mun_a_an_aper", "#ejec_cnf_mun_b_an_aper",
           "#ejec_cnf_mun_b_an_aper_prelim", "#ejecucion_cnf_mun_an_aper",
           "#ejecucion_cnf_mun_an_aper_nuevo", "#ejec_cnf_mun_a_an_aper_nuevo",
           "#ejec_cnf_mun_b_an_aper_nuevo"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


In [ ]:
# Salida agregada CNF anual
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cnf_an"))
sql_cnf_an = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_SCN,
       t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
       t1.ACTIVIDAD, t1.NIVEL, SUM(t1.Dato) AS Dato
INTO #ejecucion_cnf_an
FROM #ejecucion_cnf_an_aper t1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_SCN,
         t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
         t1.ACTIVIDAD, t1.NIVEL
"""
res = work_conn.execute(text(sql_cnf_an))
_log("CREATE #ejecucion_cnf_an", res.rowcount)
